# AIC 2026 — InfoShot++ v4.6 L25 static-state — Kaggle T4×2 end-to-end

Pipeline này hiện thực đặc tả trong `Hiểu-về-project-AIC2026.md`:

```text
video native FPS (PyAV/PTS, identity = decode-order frame index)
  ├─ native cheap sentinel: global/tile/broadcast-text/full-frame-text
  ├─ low-res Farneback motion 5 FPS
  └─ semantic probe 10 FPS: InfoShot GLRT
          ↓
  COMMON + MULTI-UNIQUE + short-shot + sentinel triggers
          ↓
  native-FPS micro-burst ±350 ms
          ↓
  BTC safety union (nếu tìm thấy mapping CSV)
          ↓
  temporal/text-aware safe dedup
          ↓
  protected coverage gap-fill (không thay thế adaptive/BTC frame)
          ↓
  L25-only verified static-state consolidation
          ↓
  JPEG + frame registry (selector embeddings là tuỳ chọn)
```

Điểm vận hành quan trọng:

- Dataset mặc định: `/kaggle/input/datasets/loinguyen57/aic-dataset-2025-s1`.
- Loại hoàn toàn `Videos_L25_a1` và `Videos_L25_b` vì trùng `Videos_L25_a`.
- Hậu tố phân mảnh `_a`, `_b`, ... trong folder/tên video được chuẩn hoá về category `Kxx/Lxx`; ví dụ `K01_a_V001` → `K01_V001` và `L26_a_V001` → `L26_V001`. Chữ trong phần khác của ID không bị xoá bừa.
- Nếu hai file còn lại cùng chuẩn hoá thành một `video_id`, bước audit sẽ **dừng** để tránh ghi đè.
- Một lượt PyAV decode đầu vừa tính sentinel native vừa encode SigLIP probe 10 FPS. Cache có fingerprint cho phép tune selection mà không encode video lại.
- Coverage chỉ chèn frame chất lượng từ probe khi output còn gap >0,5 giây; nó chạy **sau dedup**, không được ghi đè COMMON/UNIQUE/text/BTC frame.
- Riêng L25, verified static slide được nén sau coverage bằng local grayscale/edge tiles;
  dynamic interval vẫn giữ gap ≤0,5 giây, static run giữ đầu/cuối/best/protected và
  heartbeat tối đa 3,6 giây.
- Text toàn cục ưu tiên broadcast bands, nhưng localized-text trigger, text hash và text density dùng **toàn frame** để không bỏ sót slide/giấy tờ/biển báo ở giữa ảnh.
- Extraction stage không chạy PE-G14; PE/OCR/OD/caption nên chạy riêng sau khi keyframe set đã freeze. Farneback motion vẫn bật 5 FPS theo cấu hình production hiện tại.
- Pipeline nhận đủ `K01..K20` và `L21..L30`. Full-corpus batch bị chặn nếu audit
  không khớp `expected_total_videos`; session chọn subset category chỉ cần đủ category
  được yêu cầu, còn audit đủ 1.478 video được bắt buộc ở merge cuối.
- 1.478 video không vừa một Kaggle session/output quota. Mỗi session chọn category
  bằng `session_categories`, ví dụ `"L21,L22,L23"`; output được tách theo `session_id`.
  Commit mỗi output thành Kaggle Dataset, attach các dataset shard vào một notebook
  merge, rồi đặt `run_mode="merge"`. Resume trong từng session dựa trên `_SUCCESS.json`.
- Mặc định `run_mode="smoke"`: chạy trọn vẹn **mọi video trong**
  `smoke_video_paths`, rồi mở gallery phân trang riêng cho từng video ngay trong cell output. Sau khi preview ổn,
  đổi thành `"batch"`.

InfoShot gốc: [arXiv 2603.17374](https://arxiv.org/abs/2603.17374). PE dùng implementation chính thức của Meta: [facebookresearch/perception_models](https://github.com/facebookresearch/perception_models).


## 1. Cấu hình duy nhất cần chỉnh

Mặc định dùng **SigLIP** cho probe vì mạnh hơn ResNet ở transient/PiP trong ablation của InfoShot. Wrapper xử lý cả API `transformers` cũ trả Tensor và API mới trả `BaseModelOutputWithPooling`. Extraction stage lưu exact SigLIP embeddings; PE-Core-G14-448 được để `backend="none"` và nên chạy thành stage riêng trên GPU mạnh sau khi chốt keyframe.


In [ ]:
CONFIG = {
    "version": "infoshootpp-aic-v4.6-l25-static-state",
    # Giữ nguyên fingerprint pass 1 của v4.4: sửa boundary chỉ nằm sau cache.
    "pass1_version": "infoshootpp-aic-v4.4-gpu-throughput",
    "data_root": "/kaggle/input/datasets/loinguyen57/aic-dataset-2025-s1",
    # Mỗi session sẽ ghi vào output_root/session_id, không đè output session khác.
    "output_root": "/kaggle/working/infoshootpp",
    "video_extensions": [".mp4", ".mkv", ".avi", ".mov", ".webm", ".m4v"],
    "categories": (
        [f"K{i:02d}" for i in range(1, 21)]
        + [f"L{i:02d}" for i in range(21, 31)]
    ),
    "expected_total_videos": 1478,       # full K01..K20 + L21..L30 sau exclusions
    # Áp dụng cứng cho full-corpus batch. Scoped category session được audit ở merge cuối.
    "enforce_expected_total_before_batch": True,
    "exclude_dir_names": ["Videos_L25_a1", "Videos_L25_b"],
    "partition_letters": "abcdefghijklmnopqrstuvwxyz",

    # Chọn category cho SESSION NÀY. Nhận CSV string hoặc list.
    # Ví dụ: "L21,L22,L23" hoặc ["L21", "L22", "L23"]. Rỗng = toàn bộ.
    "session_categories": "L21,L22,L23",
    # Lọc thêm partition folder sau category: "d,e" sẽ chỉ lấy Videos_L26_d/e.
    # Nhận CSV string/list; rỗng = mọi partition. Dùng "main" cho folder không hậu tố.
    "session_partitions": "",
    "session_name": None,               # None = tự tạo từ category/range/shard

    # Chia nhỏ thêm bên trong các category đã chọn nếu một ca vẫn quá lớn.
    # range_end=None nghĩa là đến hết danh sách session; shard chia round-robin.
    "range_start": 0,
    "range_end": None,
    "num_shards": 1,                    # có thể chia deterministic shard giữa nhiều notebook/session
    "shard_index": 0,
    "skip_existing": True,
    "cache_pass1": True,

    # Một lượt decode: sentinel native + semantic probe.
    "probe_fps": 10.0,
    "sentinel_width": 320,
    "sentinel_grid": [8, 8],
    "sentinel_top_tiles": 4,
    "text_grid": [12, 20],             # localized subtitle/scoreboard/speedometer signal
    "text_top_tiles": 3,
    "rolling_window_sec": 2.0,
    "motion_fps": 5.0,                  # giữ ON theo yêu cầu; đặt 0 để ablation/tăng tốc
    "probe_cache_dtype": "float16",

    # Throughput: PyAV decode producer overlap với Torch-CUDA native consumer.
    "performance": {
        "native_backend": "torch_cuda", # torch_cuda | opencv_cpu
        "native_batch_frames": 768,
        "min_native_batch_frames": 192,
        "auto_reduce_native_batch_on_oom": True,
        "decode_prefetch_batches": 2,
        "native_device": "cuda:0",
        "gpu_siglip_preprocess": True,
        "gpu_semantic_glrt": True,
        "fallback_to_cpu": True,
        "opencv_threads": 2,
        "profile_stages": True,
    },

    # Selector cho InfoShot. SigLIP ưu tiên recall; ResNet-50 nhanh hơn để ablation.
    "selector": {
        "backend": "siglip",           # siglip | resnet50 | pe
        "model_name": "google/siglip-base-patch16-224",
        "pe_config": "PE-Core-B16-224",
        "batch_per_gpu": 128,
        "min_batch_per_gpu": 16,
        "auto_reduce_batch_on_oom": True,
        "allow_untrained": False,
    },

    # InfoShot GLRT/content-driven segmentation.
    "segment": {
        "min_probe_frames": 3,
        "max_probe_frames": 300,
        "window_min": 3,
        "window_max": 15,
        "boundary_z": 3.0,
        "boundary_min_raw": 0.025,
        "force_global_z": 5.0,
        "merge_boundary_ms": 200,
        "use_transnetv2": False,
        "transnet_threshold": 0.5,
    },

    "infoshoot": {
        "neighborhood_k": 1,
        "lambda_common": 0.7,
        "alpha_unique": 0.5,
        "unique_z": 2.5,
        "unique_min_score": 0.55,
        "temporal_nms_ms": 250,
        "keep_primary_unique": True,    # luôn có UNIQUE khác COMMON trong segment L>=3
    },

    # Native sentinel: robust z + absolute floor để không trigger nhiễu số học ở cảnh tĩnh.
    "sentinel": {
        "global_z": 3.5, "global_min_raw": 0.030,
        "local_z": 3.5,  "local_min_raw": 0.055,
        "text_z": 3.0,   "text_min_raw": 0.025,
        "text_local_z": 3.0, "text_local_min_raw": 0.060,
        "motion_z": 3.5, "motion_min_raw": 0.65,
        "extreme_z": 6.0,
    },

    "microburst": {
        "radius_ms": 350,
        "retain_max": 2,
        "common_radius_ms": 100,
        "boundary_radius_ms": 250,
        "second_min_score_ratio": 0.80,
        "short_segment_ms": 800,
        "very_short_segment_ms": 500,
        "complement_phash_hamming": 6,
        "complement_text_hamming": 6,
    },

    "btc_union": {
        "enabled": False,               # pure InfoShot++: không union keyframe/mapping BTC có sẵn
        "require_mapping": False,        # chỉ dùng BTC khi CSV có frame_idx/pts_time
        "prefer_btc": True,
    },

    "dedup": {
        "max_temporal_distance_ms": 1000,
        "semantic_cosine": 0.985,
        "phash_hamming": 4,
        "text_hash_hamming": 4,
        "text_density_delta": 0.015,
    },

    # High-recall safety net từ v4, triển khai dạng gap-fill sau adaptive dedup.
    # target_step_sec < max_gap_sec để hấp thụ sai số timestamp/probe rounding.
    "coverage": {
        "enabled": True,
        "max_gap_sec": 0.50,
        "target_step_sec": 0.40,
        "quality_weight": 0.15,
        "verify_tolerance_sec": 0.02,
    },

    # L25 lecture/static-slide only: compress verified identical visual states after
    # coverage. Local tile guards protect small handwriting/circle/highlight changes.
    "l25_static_consolidation": {
        "enabled": True,
        "categories": ["L25"],
        "analysis_width": 640,
        "grid": [12, 20],
        "top_tiles": 1,
        "min_run_frames": 5,
        "static_min_sec": 1.5,
        "heartbeat_sec": 3.0,
        "max_static_gap_sec": 3.6,
        "max_input_gap_sec": 0.60,
        "phash_hamming": 2,
        "text_hash_hamming": 2,
        "text_density_delta": 0.005,
        "semantic_cosine": 0.995,
        "global_gray_delta": 0.012,
        "local_gray_delta": 0.025,
        "global_edge_delta": 0.010,
        "local_edge_delta": 0.025,
        "motion_max_raw": 0.35,
        "protected_sources": [
            "btc", "text_event", "text_local_event", "semantic_boundary",
            "short_shot", "motion_event",
        ],
        "fallback_on_verify_failure": True,
    },

    # PE/OCR/OD/caption là stage riêng sau khi keyframe set đã freeze.
    "final_embeddings": {
        "backend": "none",              # extraction: none; stage sau có thể pe | selector
        "pe_config": "PE-Core-G14-448",
        "batch_per_gpu": 1,
        "required": False,
    },

    # Max-throughput extraction: không re-encode toàn bộ retained frame lần thứ hai.
    # Bật lại save_selector_embeddings nếu cần benchmark/debug SigLIP trực tiếp.
    "storage": {
        "batch_registry_only": False,
        "save_selector_embeddings": False,
        "save_final_embeddings": False,
        "keep_pass1_cache": True,
        "export_registry_zip": True,
    },

    # Kaggle output thường bị giới hạn khoảng 20 GB. JPEG vốn đã nén nên ZIP chủ yếu
    # giảm file-count; quota_gb-reserve mới là hàng rào thật sự chống vượt quota.
    "session_storage": {
        "enabled": True,
        "archive_keyframes": True,
        "archive_trigger_gb": 12.0,
        "zip_compresslevel": 1,          # nhanh; JPEG gần như không lợi ở level cao
        "delete_images_after_verify": True,
        "delete_pass1_cache_after_success": True,
        "quota_gb": 20.0,
        "reserve_for_next_video_gb": 2.0, # dừng trước video kế khi session đạt 18 GiB
    },

    "save_max_side": 1280,
    "jpg_quality": 92,
    "run_mode": "batch",                # audit | smoke | batch | merge
    # Smoke chạy FULL tuần tự tất cả video trong list. Có thể để list chỉ một phần tử.
    "smoke_video_paths": [
        (
            "/kaggle/input/datasets/loinguyen57/aic-dataset-2025-s1/"
            "Videos_L30_a/video/L30_V001.mp4"
        ),
        (
            "/kaggle/input/datasets/loinguyen57/aic-dataset-2025-s1/"
            "Videos_L25_a/video/L25_V013.mp4"
        ),
        (
            "/kaggle/input/datasets/loinguyen57/aic-dataset-2025-s1/"
            "Videos_L27_a/video/L27_V013.mp4"
        ),
    ],
    "smoke_preview_page_size": 36,       # số ảnh mỗi trang trong cell output
    "smoke_preview_columns": 6,
    "smoke_static_sample": 120,          # contact sheet sample đều + gap report
    # Deep review source frames quanh timestamp nghi ngờ; [] để tắt.
    "smoke_inspect_times": [201.0],       # 201 s ≈ 03:21; đổi theo video đang test
    "smoke_inspect_radius_sec": 2.0,
    "smoke_inspect_fps": 5.0,
    "smoke_inspect_max_side": 960,

    # Merge các Kaggle Dataset output của nhiều session. Có thể trỏ tới dataset root;
    # notebook sẽ tự tìm run_manifest.json bên dưới.
    "merge": {
        "input_roots": [
            # "/kaggle/input/infoshootpp-l21-l23",
            # "/kaggle/input/infoshootpp-l24-l26",
        ],
        "output_root": "/kaggle/working/infoshootpp_merged",
        # False (khuyên dùng): global registry trỏ tới ảnh trong attached shard,
        # không copy hàng trăm GB JPEG. True: copy ảnh/embedding vào output merge.
        "copy_assets": False,
        "strict_pipeline_signature": True,
        "strict_session_complete": True,
        # None = phải đủ toàn bộ CONFIG["categories"]. Có thể đặt CSV/list subset.
        "expected_categories": None,
        "expected_total_videos": 1478,
        "enforce_expected_total": True,
    },
}

# Chỉ bật install cho backend thực sự dùng. Kaggle cần Internet ON ở lần tải model đầu.
NEEDS_MODELS = CONFIG["run_mode"] not in {"audit", "merge"}
INSTALL_SIGLIP = NEEDS_MODELS and CONFIG["selector"]["backend"] == "siglip"
INSTALL_PE = (
    NEEDS_MODELS and (
    CONFIG["selector"]["backend"] == "pe"
    or CONFIG["final_embeddings"]["backend"] == "pe"
    )
)
INSTALL_TRANSNETV2 = NEEDS_MODELS and CONFIG["segment"]["use_transnetv2"]


## 2. Dependencies

Kaggle đã có PyTorch/torchvision. Cell này cài PyAV và các nhánh model cần dùng, không upgrade torch/CUDA.


In [ ]:
import os, sys, subprocess
from pathlib import Path

def pip_install(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

pip_install("av>=12", "opencv-python-headless>=4.10", "scipy>=1.11", "pyarrow>=15")

if INSTALL_SIGLIP:
    pip_install("transformers>=4.55,<6", "accelerate>=1.0", "safetensors")

if INSTALL_TRANSNETV2:
    pip_install("transnetv2-pytorch")

PE_REPO = Path("/kaggle/working/perception_models")
if INSTALL_PE:
    try:
        if not PE_REPO.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/facebookresearch/perception_models.git", str(PE_REPO)],
                check=True,
            )
        pip_install("-e", str(PE_REPO), "--no-deps")
        pip_install("einops", "timm", "ftfy", "regex", "iopath")
        if str(PE_REPO) not in sys.path:
            sys.path.insert(0, str(PE_REPO))
    except Exception as exc:
        pe_required = (
            CONFIG["selector"]["backend"] == "pe"
            or CONFIG["final_embeddings"]["required"]
        )
        if pe_required:
            raise
        print(f"WARNING: optional PE install failed: {type(exc).__name__}: {exc}")
        print("Pipeline sẽ tiếp tục với exact selector embeddings.")

print("Optional dependencies ready.")


## 3. Imports, output layout và helper chung


In [ ]:
import csv
import gc
import hashlib
import json
import math
import re
import shutil
import subprocess
import threading
import time
import traceback
import warnings
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from fractions import Fraction
from pathlib import Path
from typing import Any, Iterable, Optional

# Phải đặt trước import cv2 để ép software decode cho AV1 trên Kaggle/T4.
os.environ["OPENCV_FFMPEG_CAPTURE_OPTIONS"] = "hwaccel;none"

import cv2
import av
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from IPython.display import display
from scipy.ndimage import median_filter
from tqdm.auto import tqdm

torch.backends.cuda.matmul.allow_tf32 = True
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

warnings.filterwarnings("ignore")
cv2.setNumThreads(max(1, int(CONFIG["performance"]["opencv_threads"])))
torch.set_grad_enabled(False)

def parse_category_spec(value):
    if value is None:
        return []
    if isinstance(value, str):
        tokens = re.split(r"[,;\s]+", value.strip()) if value.strip() else []
    elif isinstance(value, (list, tuple, set)):
        tokens = [str(x).strip() for x in value]
    else:
        raise TypeError("session_categories phải là CSV string hoặc list/tuple/set")
    tokens = [x.upper() for x in tokens if x]
    tokens = list(dict.fromkeys(tokens))
    unknown = sorted(set(tokens)-set(CONFIG["categories"]))
    if unknown:
        raise ValueError(f"session_categories không hợp lệ: {unknown}")
    order = {name:i for i,name in enumerate(CONFIG["categories"])}
    return sorted(tokens, key=order.__getitem__)

def parse_partition_spec(value):
    if value is None:
        return []
    if isinstance(value, str):
        tokens = re.split(r"[,;\s]+", value.strip()) if value.strip() else []
    elif isinstance(value, (list, tuple, set)):
        tokens = [str(x).strip() for x in value]
    else:
        raise TypeError("session_partitions phải là CSV string hoặc list/tuple/set")
    tokens = [x.lower().lstrip("_") for x in tokens if x]
    tokens = list(dict.fromkeys(tokens))
    allowed = {"main", *list(str(CONFIG["partition_letters"]).lower())}
    unknown = sorted(set(tokens)-allowed)
    if unknown:
        raise ValueError(f"session_partitions không hợp lệ: {unknown}")
    return sorted(tokens, key=lambda x:(x != "main", x))

def make_session_id(categories, partitions=None):
    explicit = CONFIG.get("session_name")
    if explicit:
        base = str(explicit)
    else:
        base = "-".join(categories).lower() if categories else "all-categories"
        if partitions:
            base += "__p-" + "-".join(partitions)
    if int(CONFIG.get("range_start", 0)) != 0 or CONFIG.get("range_end") is not None:
        end = "end" if CONFIG.get("range_end") is None else str(CONFIG["range_end"])
        base += f"__r{int(CONFIG.get('range_start',0))}-{end}"
    if int(CONFIG.get("num_shards", 1)) > 1:
        base += f"__s{int(CONFIG['shard_index']):02d}of{int(CONFIG['num_shards']):02d}"
    session_id = re.sub(r"[^a-zA-Z0-9._-]+", "-", base).strip("-_.").lower()
    if not session_id:
        raise ValueError("session_name tạo ra session_id rỗng")
    return session_id

DATA_ROOT = Path(CONFIG["data_root"])
SESSION_CATEGORIES = parse_category_spec(CONFIG.get("session_categories"))
SESSION_PARTITIONS = parse_partition_spec(CONFIG.get("session_partitions"))
SESSION_ID = make_session_id(SESSION_CATEGORIES, SESSION_PARTITIONS)
BASE_OUTPUT_ROOT = Path(CONFIG["output_root"])
SESSION_OUT = BASE_OUTPUT_ROOT / SESSION_ID
MERGE_OUT = Path(CONFIG["merge"]["output_root"])
OUT = MERGE_OUT if CONFIG["run_mode"] == "merge" else SESSION_OUT
for name in ["cache/probe", "cache/sentinel", "keyframes", "archives", "registry",
             "map-keyframes", "selector-embeddings", "final-embeddings", "runs", "errors"]:
    (OUT / name).mkdir(parents=True, exist_ok=True)

print("Run mode:", CONFIG["run_mode"])
print("Session ID:", SESSION_ID)
print("Session categories:", SESSION_CATEGORIES or "ALL")
print("Session partitions:", SESSION_PARTITIONS or "ALL")
print("Output:", OUT)
if CONFIG["run_mode"] == "batch" and CONFIG["session_storage"].get("enabled", True):
    storage_cfg = CONFIG["session_storage"]
    trigger_gb = float(storage_cfg["archive_trigger_gb"])
    stop_line_gb = (
        float(storage_cfg["quota_gb"])
        - float(storage_cfg["reserve_for_next_video_gb"])
    )
    if not (0 < trigger_gb < stop_line_gb):
        raise ValueError(
            "Cần 0 < archive_trigger_gb < quota_gb-reserve_for_next_video_gb"
        )
    print(f"Storage guard: archive từ {trigger_gb:.1f} GiB, "
          f"dừng trước video kế ở {stop_line_gb:.1f} GiB")

DEVICES = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
print("Devices:", DEVICES)
for d in DEVICES:
    if d != "cpu":
        print(" ", d, torch.cuda.get_device_name(int(d.split(":")[1])))

def log(*items):
    print("[InfoShot++]", *items, flush=True)

def stable_json(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def fingerprint(obj: Any, n: int = 12) -> str:
    return hashlib.sha256(stable_json(obj).encode("utf-8")).hexdigest()[:n]

def atomic_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(tmp, path)

def sec_to_hms(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:06.3f}"

def natural_video_key(video_id: str):
    m = re.match(r"([A-Z]+)(\d+)_V(\d+)", video_id.upper())
    return (m.group(1), int(m.group(2)), int(m.group(3))) if m else (video_id, 0, 0)

PIPELINE_SIGNATURE = fingerprint({
    "version": CONFIG["version"],
    "probe_fps": CONFIG["probe_fps"],
    "native_scan": {
        "sentinel_width": CONFIG["sentinel_width"],
        "sentinel_grid": CONFIG["sentinel_grid"],
        "sentinel_top_tiles": CONFIG["sentinel_top_tiles"],
        "text_grid": CONFIG["text_grid"],
        "text_top_tiles": CONFIG["text_top_tiles"],
        "rolling_window_sec": CONFIG["rolling_window_sec"],
        "motion_fps": CONFIG["motion_fps"],
    },
    "performance": CONFIG["performance"],
    "selector": CONFIG["selector"],
    "segment": CONFIG["segment"],
    "infoshoot": CONFIG["infoshoot"],
    "sentinel": CONFIG["sentinel"],
    "microburst": CONFIG["microburst"],
    "dedup": CONFIG["dedup"],
    "coverage": CONFIG["coverage"],
    "l25_static_consolidation": CONFIG["l25_static_consolidation"],
    "btc_union": CONFIG["btc_union"],
    "final_embeddings": CONFIG["final_embeddings"],
    "save_max_side": CONFIG["save_max_side"],
    "jpg_quality": CONFIG["jpg_quality"],
    "storage": CONFIG["storage"],
})
print("Pipeline signature:", PIPELINE_SIGNATURE)


## 4. Discover + audit dữ liệu

Hai folder duplicate L25 bị loại theo **path component chính xác**. Chuẩn hoá chỉ tác động marker phân mảnh đứng giữa category và `_V...`; không xoá chữ cái trong tên tuỳ tiện.

Discovery dùng layout cố định `Videos_<Kxx/Lxx>[_partition]/video/*`: chỉ quét hai cấp thư mục cần thiết, **không duyệt đệ quy** qua toàn bộ dataset/Keyframes.


In [ ]:
_CANONICAL_VIDEO_RE = re.compile(r"(?i)([KL]\d{2})(?:_?([a-z])\d*)?_V(\d+)")
_VIDEO_FOLDER_RE = re.compile(r"(?i)^Videos_([KL]\d{2})(?:_([a-z]\d*))?$")

def canonical_video_id(value: str) -> str:
    stem = Path(str(value)).stem.replace("-", "_")
    m = _CANONICAL_VIDEO_RE.search(stem)
    if m:
        return f"{m.group(1).upper()}_V{m.group(3)}"
    # Fallback có kiểm soát: chỉ bỏ marker partition ngay sau Kxx/Lxx.
    stem = re.sub(r"(?i)^([KL]\d{2})_[a-z]\d*(?=_|$)", r"\1", stem)
    return stem.upper()

def source_partition(path: Path) -> str:
    for part in path.parts:
        m = re.fullmatch(r"(?i)Videos_([KL]\d{2})(?:_([a-z]\d*))?", part)
        if m:
            return (m.group(2) or "main").lower()
    return "unknown"

def excluded_reason(path: Path) -> Optional[str]:
    excluded = {x.casefold() for x in CONFIG["exclude_dir_names"]}
    for part in path.parts:
        if part.casefold() in excluded:
            return part
    return None

def discover_videos(data_root: Path = DATA_ROOT):
    if not data_root.exists():
        raise FileNotFoundError(
            f"Không thấy dataset: {data_root}. Hãy Attach dataset hoặc sửa CONFIG['data_root']."
        )
    exts = {x.lower() for x in CONFIG["video_extensions"]}
    allowed_categories = {str(x).upper() for x in CONFIG["categories"]}
    excluded_names = {x.casefold() for x in CONFIG["exclude_dir_names"]}
    records = []
    scanned_folders = 0

    # Fast path theo layout Kaggle AIC:
    # DATA_ROOT/Videos_Kxx[_partition]/video/*.mp4
    # Không dùng os.walk/rglob, nên tuyệt đối không chạm vào cây Keyframes_*.
    with os.scandir(data_root) as root_entries:
        for folder_entry in root_entries:
            if not folder_entry.is_dir():
                continue
            folder_match = _VIDEO_FOLDER_RE.fullmatch(folder_entry.name)
            if not folder_match:
                continue

            folder_category = folder_match.group(1).upper()
            if folder_category not in allowed_categories:
                continue
            partition = (folder_match.group(2) or "main").lower()
            reason = folder_entry.name if folder_entry.name.casefold() in excluded_names else None
            folder = Path(folder_entry.path)
            video_subdir = folder / "video"
            scan_dir = video_subdir if video_subdir.is_dir() else folder
            scanned_folders += 1

            # Chỉ duyệt file trực tiếp trong /video (hoặc trực tiếp trong Videos_* nếu
            # dataset không có lớp /video); không đi xuống thư mục con nào khác.
            with os.scandir(scan_dir) as video_entries:
                for file_entry in video_entries:
                    if not file_entry.is_file() or Path(file_entry.name).suffix.lower() not in exts:
                        continue
                    path = Path(file_entry.path)
                    vid = canonical_video_id(file_entry.name)
                    category = vid.split("_", 1)[0]
                    records.append({
                        "path": str(path), "raw_stem": path.stem, "video_id": vid,
                        "category": category, "partition": partition,
                        "excluded": reason is not None, "exclude_reason": reason or "",
                        "size_bytes": int(file_entry.stat().st_size),
                    })
    raw = pd.DataFrame(records)
    if raw.empty:
        raise RuntimeError(
            f"Không tìm thấy video theo layout Videos_Kxx/Lxx[/video] trong {data_root}. "
            "Kiểm tra CONFIG['data_root'] và cấu trúc dataset."
        )

    print(
        f"Fast discovery (non-recursive): {scanned_folders:,} Videos_* folder, "
        f"{len(raw):,} video file"
    )

    kept = raw[~raw.excluded].copy()
    kept = kept[kept.category.isin(CONFIG["categories"])].copy()

    collisions = kept.groupby("video_id").filter(lambda x: len(x) > 1)
    if not collisions.empty:
        display(collisions[["video_id", "raw_stem", "partition", "path"]].sort_values("video_id"))
        raise RuntimeError(
            "Canonical video_id bị trùng sau khi đã exclude L25_a1/L25_b. "
            "Không chạy tiếp để tránh ghi đè; kiểm tra bảng collision ở trên."
        )

    kept["_sort"] = kept.video_id.map(natural_video_key)
    kept = kept.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)
    return raw, kept

def audit_dataset(raw: pd.DataFrame, kept: pd.DataFrame):
    print(f"Tổng file video thấy được: {len(raw):,}")
    excluded = raw[raw.excluded]
    print(f"Đã exclude duplicate L25: {len(excluded):,}")
    if len(excluded):
        display(excluded[["raw_stem", "partition", "exclude_reason", "path"]].head(30))
    print(f"Video hợp lệ sau filter/normalize: {len(kept):,}")
    expected = CONFIG.get("expected_total_videos")
    count_ok = expected is None or len(kept) == int(expected)
    categories_found = sorted(kept.category.unique())
    missing_categories = sorted(set(CONFIG["categories"]) - set(categories_found))
    categories_ok = not missing_categories
    audit_ok = bool(count_ok and categories_ok)
    if expected is not None:
        mark = "✓" if audit_ok else "⚠"
        print(f"{mark} Full-dataset audit: found={len(kept):,}, expected={int(expected):,}")
        if missing_categories:
            print("Missing categories:", missing_categories)
        if not audit_ok:
            if SESSION_CATEGORIES and set(SESSION_CATEGORIES) != set(CONFIG["categories"]):
                print("Full-corpus chưa đủ; scoped category session vẫn có thể chạy. "
                      "Merge cuối sẽ bắt buộc audit đủ tổng video/category.")
            else:
                print("Full-corpus batch sẽ bị chặn; kiểm tra dataset attach và exclusions.")
    summary = kept.groupby(["category", "partition"]).size().rename("videos").reset_index()
    display(summary)
    display(kept[["video_id", "raw_stem", "category", "partition", "path"]].head(20))
    assert kept.video_id.is_unique
    assert not any(kept.path.str.contains(r"Videos_L25_(?:a1|b)(?:/|$)", case=False, regex=True))
    return summary, {
        "found": int(len(kept)), "expected": int(expected) if expected is not None else None,
        "count_ok": bool(count_ok), "categories_ok": bool(categories_ok),
        "audit_ok": audit_ok, "categories_found": categories_found,
        "missing_categories": missing_categories,
    }

def select_video_groups(all_videos: pd.DataFrame):
    groups = SESSION_CATEGORIES
    category_selected = all_videos if not groups else all_videos[all_videos.category.isin(groups)]
    missing = sorted(set(groups)-set(category_selected.category.unique()))
    if missing:
        raise RuntimeError(f"Session yêu cầu category nhưng dataset không có: {missing}")
    selected = category_selected
    if SESSION_PARTITIONS:
        selected = selected[selected.partition.astype(str).str.lower().isin(SESSION_PARTITIONS)]
    selected = selected.reset_index(drop=True)
    missing_partitions = sorted(set(SESSION_PARTITIONS)-set(selected.partition.astype(str).str.lower()))
    if missing_partitions:
        raise RuntimeError(
            f"Session yêu cầu partition nhưng category đã chọn không có: {missing_partitions}"
        )
    print(f"Session {SESSION_ID}: {len(selected):,} video trước range/shard")
    if len(selected):
        display(selected.groupby(["category","partition"]).size().rename("videos").reset_index())
    return selected

def assert_full_dataset_ready_for_batch():
    if not CONFIG.get("enforce_expected_total_before_batch", True):
        return
    scoped_session = bool(
        SESSION_CATEGORIES and set(SESSION_CATEGORIES) != set(CONFIG["categories"])
    )
    if scoped_session:
        found = set(VIDEOS_ALL.category.unique())
        missing_requested = sorted(set(SESSION_CATEGORIES)-found)
        if missing_requested:
            raise RuntimeError(
                f"SESSION BLOCKED: thiếu category được yêu cầu: {missing_requested}"
            )
        log(f"Scoped session audit OK: {SESSION_CATEGORIES}; "
            "full-corpus count sẽ kiểm tra ở merge cuối")
        return
    if not DATASET_AUDIT["audit_ok"]:
        raise RuntimeError(
            "BATCH BLOCKED: full-dataset audit không đạt: "
            f"found={DATASET_AUDIT['found']}, expected={DATASET_AUDIT['expected']}, "
            f"missing_categories={DATASET_AUDIT['missing_categories']}."
        )

if CONFIG["run_mode"] == "merge":
    # Merge chỉ cần các output shard đã attach, không cần attach lại video nguồn/model.
    VIDEOS_RAW = VIDEOS_ALL = VIDEOS = pd.DataFrame(
        columns=["video_id", "raw_stem", "category", "partition", "path"]
    )
    DATASET_SUMMARY = pd.DataFrame()
    DATASET_AUDIT = {"audit_ok": True, "merge_mode": True}
    print("Merge mode: bỏ qua discover/audit video nguồn.")
else:
    VIDEOS_RAW, VIDEOS_ALL = discover_videos()
    DATASET_SUMMARY, DATASET_AUDIT = audit_dataset(VIDEOS_RAW, VIDEOS_ALL)
    VIDEOS = select_video_groups(VIDEOS_ALL)


## 5. Video metadata, hashes, quality và data classes


In [ ]:
@dataclass
class VideoMeta:
    path: str
    video_id: str
    category: str
    fps: float
    total_frames: int
    duration_sec: float
    width: int
    height: int
    codec: str = "?"

@dataclass
class Event:
    frame_idx: int
    start_frame: int
    end_frame: int
    pts_time: Optional[float] = None
    start_time: Optional[float] = None
    end_time: Optional[float] = None
    radius_ms: Optional[int] = None
    sources: set[str] = field(default_factory=set)
    priority: int = 2               # P0=0, P1=1, P2=2
    scores: dict[str, float] = field(default_factory=dict)
    segment_id: int = -1

@dataclass
class FrameCandidate:
    frame_idx: int
    pts_time: float
    segment_id: int
    sources: set[str] = field(default_factory=set)
    priority: int = 2
    scores: dict[str, float] = field(default_factory=dict)
    quality: float = 0.0
    phash: int = 0
    text_hash: int = 0
    text_density: float = 0.0
    embedding: Optional[np.ndarray] = field(default=None, repr=False)
    btc_id: Optional[str] = None

def parse_rate(value: str) -> float:
    try:
        return float(Fraction(value))
    except Exception:
        try: return float(value)
        except Exception: return 0.0

def probe_video(path: str, video_id: Optional[str] = None) -> VideoMeta:
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=avg_frame_rate,r_frame_rate,nb_frames,width,height,codec_name,duration",
        "-show_entries", "format=duration", "-of", "json", path,
    ]
    info = {}
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, check=True)
        payload = json.loads(proc.stdout)
        info = (payload.get("streams") or [{}])[0]
        fmt = payload.get("format") or {}
        duration = float(info.get("duration") or fmt.get("duration") or 0)
        fps = parse_rate(info.get("avg_frame_rate") or info.get("r_frame_rate") or "0")
        total = int(info.get("nb_frames") or 0)
        if total <= 0 and duration > 0 and fps > 0:
            total = int(round(duration * fps))
    except Exception:
        cap = cv2.VideoCapture(path)
        fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        duration = total / fps if fps > 0 else 0
        info = {"width": cap.get(cv2.CAP_PROP_FRAME_WIDTH),
                "height": cap.get(cv2.CAP_PROP_FRAME_HEIGHT), "codec_name": "?"}
        cap.release()
    if fps <= 0 or not np.isfinite(fps):
        raise RuntimeError(f"FPS không hợp lệ: {path}")
    return VideoMeta(
        path=str(path), video_id=video_id or canonical_video_id(path),
        category=(video_id or canonical_video_id(path)).split("_", 1)[0],
        fps=fps, total_frames=total, duration_sec=duration or total / fps,
        width=int(info.get("width") or 0), height=int(info.get("height") or 0),
        codec=str(info.get("codec_name") or "?"),
    )

def resize_width(image: np.ndarray, width: int) -> np.ndarray:
    h, w = image.shape[:2]
    if w <= width: return image
    scale = width / float(w)
    return cv2.resize(image, (width, max(1, int(round(h * scale)))), interpolation=cv2.INTER_AREA)

def resize_max_side(image: np.ndarray, max_side: int) -> np.ndarray:
    h, w = image.shape[:2]
    side = max(h, w)
    if side <= max_side: return image
    scale = max_side / float(side)
    return cv2.resize(image, (max(1, int(round(w * scale))), max(1, int(round(h * scale)))),
                      interpolation=cv2.INTER_AREA)

def _hash64_from_gray(gray: np.ndarray) -> int:
    small = cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    dct = cv2.dct(small)[:8, :8]
    threshold = float(np.median(dct[1:, :]))
    bits = (dct > threshold).reshape(-1)
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return int(value)

def hamming64(a: int, b: int) -> int:
    return int(int(a) ^ int(b)).bit_count()

def _binary_text_edges(gray: np.ndarray) -> np.ndarray:
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy)
    threshold = max(24.0, float(np.percentile(mag, 75)))
    edge = (mag >= threshold).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 1))
    return cv2.morphologyEx(edge, cv2.MORPH_CLOSE, kernel)

def broadcast_text_edge_map(gray: np.ndarray) -> np.ndarray:
    """Ưu tiên scoreboard/lower-third/ticker ở top 20% và bottom 40%."""
    h = gray.shape[0]
    bands = np.concatenate(
        [gray[:max(1, int(0.20*h))], gray[min(h-1, int(0.60*h)):]], axis=0
    )
    return _binary_text_edges(bands)

def full_text_edge_map(gray: np.ndarray) -> np.ndarray:
    """Toàn frame cho localized trigger, text hash và dedup protection."""
    return _binary_text_edges(gray)

def visual_signatures(image_bgr: np.ndarray):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    edges = full_text_edge_map(gray)
    return _hash64_from_gray(gray), _hash64_from_gray(edges), float((edges > 0).mean())

def frame_quality(image_bgr: np.ndarray):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    sharp_raw = float(cv2.Laplacian(gray, cv2.CV_32F).var())
    sharp = float(np.clip(np.log1p(sharp_raw) / np.log1p(1500.0), 0, 1))
    mean = float(gray.mean() / 255.0)
    exposure = float(np.clip(1.0 - abs(mean - 0.5) / 0.5, 0, 1))
    clipped = float(((gray < 5) | (gray > 250)).mean())
    black = float((gray < 10).mean())
    quality = float(np.clip(0.62*sharp + 0.38*exposure - 0.55*clipped - 0.35*(black > 0.92), 0, 1))
    return quality, sharp_raw, mean, clipped, black

def probe_frame_metrics(image_bgr: np.ndarray):
    """Một lần grayscale/Sobel cho toàn bộ quality + hash metric của probe frame."""
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    sharp_raw = float(cv2.Laplacian(gray, cv2.CV_32F).var())
    sharp = float(np.clip(np.log1p(sharp_raw)/np.log1p(1500.0), 0, 1))
    mean = float(gray.mean()/255.0)
    exposure = float(np.clip(1.0-abs(mean-0.5)/0.5, 0, 1))
    clipped = float(((gray < 5) | (gray > 250)).mean())
    black = float((gray < 10).mean())
    quality = float(np.clip(
        0.62*sharp+0.38*exposure-0.55*clipped-0.35*(black > 0.92), 0, 1
    ))
    edges = full_text_edge_map(gray)
    phash = _hash64_from_gray(gray)
    text_hash = _hash64_from_gray(edges)
    text_density = float((edges > 0).mean())
    return quality, sharp_raw, phash, text_hash, text_density

def minmax(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, np.float32)
    lo, hi = float(np.min(x)), float(np.max(x))
    return np.zeros_like(x) if hi - lo < 1e-9 else (x - lo) / (hi - lo)

def robust_z_global(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, np.float32)
    med = float(np.median(x))
    mad = float(np.median(np.abs(x - med)))
    return (x - med) / (1.4826 * mad + 1e-4)


## 6. Encoder đa GPU

Hai T4 chạy hai nửa batch độc lập; không dùng `DataParallel`. Resize, BGR→RGB,
scale và normalize của SigLIP chạy ngay trên từng GPU thay vì dùng PIL/CPU. Batch mặc
định là 128 ảnh/GPU và tự giảm khi CUDA OOM. Wrapper SigLIP luôn trả Tensor trên cả
API `transformers` cũ/mới. ResNet/PE vẫn có sẵn cho ablation. Mọi vector đều
L2-normalized.


In [ ]:
class MultiDeviceEncoder:
    kind = "base"
    dim = 0
    image_size = 224

    def __init__(self, batch_per_gpu: int, min_batch_per_gpu=1,
                 auto_reduce_batch_on_oom=True):
        self.devices = DEVICES
        self.batch_per_gpu = int(batch_per_gpu)
        self.min_batch_per_gpu = max(1, int(min_batch_per_gpu))
        self.auto_reduce_batch_on_oom = bool(auto_reduce_batch_on_oom)

    @property
    def total_batch_size(self):
        return self.batch_per_gpu * len(self.devices)

    def encode(self, images_bgr: list[np.ndarray]) -> np.ndarray:
        if not images_bgr:
            return np.empty((0, self.dim), np.float32)
        outputs = []
        start = 0
        while start < len(images_bgr):
            stop = min(len(images_bgr), start+self.total_batch_size)
            try:
                outputs.append(self._encode_batch(images_bgr[start:stop]))
                start = stop
            except RuntimeError as exc:
                is_oom = "out of memory" in str(exc).lower() and torch.cuda.is_available()
                next_batch = max(self.min_batch_per_gpu, self.batch_per_gpu//2)
                if not (is_oom and self.auto_reduce_batch_on_oom and next_batch < self.batch_per_gpu):
                    raise
                log(f"CUDA OOM: giảm batch/GPU {self.batch_per_gpu} → {next_batch} rồi retry")
                self.batch_per_gpu = next_batch
                gc.collect()
                for device in self.devices:
                    if device != "cpu":
                        with torch.cuda.device(device): torch.cuda.empty_cache()
        return np.concatenate(outputs, axis=0).astype(np.float32, copy=False)

class ResNet50Encoder(MultiDeviceEncoder):
    kind = "resnet50-imagenet1k-v2"
    dim = 2048
    image_size = 224

    def __init__(self, batch_per_gpu=64, allow_untrained=False):
        super().__init__(batch_per_gpu)
        from torchvision.models import ResNet50_Weights, resnet50
        self.models = []
        for device in self.devices:
            try:
                net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
            except Exception:
                if not allow_untrained:
                    raise RuntimeError(
                        "Không tải được pretrained ResNet-50. Bật Internet hoặc attach torch cache; "
                        "không dùng random weights cho keyframe selection."
                    )
                log("WARNING: ResNet random weights; chỉ dùng debug, không dùng production.")
                net = resnet50(weights=None)
            net.fc = nn.Identity()
            dtype = torch.float16 if device != "cpu" else torch.float32
            net = net.eval().to(device=device, dtype=dtype)
            self.models.append(net)
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

    def _prepare(self, images):
        arr = np.stack([cv2.resize(im, (224, 224), interpolation=cv2.INTER_AREA)[:, :, ::-1]
                        for im in images]).copy()
        x = torch.from_numpy(arr).permute(0,3,1,2).float().div_(255.0)
        return (x - self.mean) / self.std

    def _encode_batch(self, images):
        x = self._prepare(images)
        chunks = [c for c in torch.tensor_split(x, len(self.devices)) if len(c)]
        def run(i, chunk):
            device, model = self.devices[i], self.models[i]
            dtype = torch.float16 if device != "cpu" else torch.float32
            with torch.inference_mode(), torch.autocast(
                device_type="cuda" if device != "cpu" else "cpu",
                dtype=torch.float16, enabled=device != "cpu"
            ):
                feat = model(chunk.to(device=device, dtype=dtype, non_blocking=True))
            return F.normalize(feat.float(), dim=-1).cpu().numpy()
        with ThreadPoolExecutor(max_workers=len(chunks)) as pool:
            futures = [pool.submit(run, i, chunk) for i, chunk in enumerate(chunks)]
            return np.concatenate([f.result() for f in futures], axis=0)

class SigLIPImageWrapper(nn.Module):
    """Chuẩn hoá output get_image_features của transformers v4/v5 thành Tensor [B,D]."""
    def __init__(self, base):
        super().__init__(); self.base = base

    def forward(self, pixel_values):
        if hasattr(self.base, "get_image_features"):
            out = self.base.get_image_features(pixel_values=pixel_values)
        else:
            out = self.base.vision_model(pixel_values=pixel_values)
        if torch.is_tensor(out): return out
        pool = getattr(out, "pooler_output", None)
        if torch.is_tensor(pool): return pool
        if isinstance(out, (tuple, list)):
            if len(out) > 1 and torch.is_tensor(out[1]): return out[1]
            if len(out) and torch.is_tensor(out[0]):
                return out[0].mean(dim=1) if out[0].ndim == 3 else out[0]
        last = getattr(out, "last_hidden_state", None)
        if torch.is_tensor(last): return last.mean(dim=1)
        raise TypeError(f"SigLIP output không hỗ trợ: {type(out)!r}")

class SigLIPEncoder(MultiDeviceEncoder):
    def __init__(self, model_name, batch_per_gpu=16, min_batch_per_gpu=8,
                 auto_reduce_batch_on_oom=True):
        super().__init__(batch_per_gpu, min_batch_per_gpu, auto_reduce_batch_on_oom)
        from transformers import AutoModel, AutoProcessor
        self.kind = f"siglip:{model_name}"
        self.processor = AutoProcessor.from_pretrained(model_name)
        image_processor = getattr(self.processor, "image_processor", self.processor)
        size = getattr(image_processor, "size", 224)
        if isinstance(size, dict):
            height = int(size.get("height") or size.get("shortest_edge") or 224)
            width = int(size.get("width") or size.get("shortest_edge") or height)
        else:
            height = width = int(size)
        self.input_hw = (height, width)
        self.image_size = height
        self.gpu_preprocess = bool(
            CONFIG["performance"]["gpu_siglip_preprocess"] and torch.cuda.is_available()
        )
        mean = getattr(image_processor, "image_mean", [0.5, 0.5, 0.5])
        std = getattr(image_processor, "image_std", [0.5, 0.5, 0.5])
        self.norm = {}
        self.models = []
        for device in self.devices:
            dtype = torch.float16 if device != "cpu" else torch.float32
            base = AutoModel.from_pretrained(model_name, torch_dtype=dtype).eval()
            model = SigLIPImageWrapper(base).eval().to(device)
            self.models.append(model)
            self.norm[device] = (
                torch.tensor(mean, device=device, dtype=torch.float32).view(1,3,1,1),
                torch.tensor(std, device=device, dtype=torch.float32).view(1,3,1,1),
            )
        probe = self.encode([np.zeros((32,32,3), np.uint8)])
        self.dim = probe.shape[1]

    def _encode_batch(self, images):
        if self.gpu_preprocess:
            arr = np.ascontiguousarray(np.stack(images, axis=0))
            px = torch.from_numpy(arr).permute(0,3,1,2)
            if torch.cuda.is_available(): px = px.pin_memory()
        else:
            pil = [Image.fromarray(cv2.cvtColor(x, cv2.COLOR_BGR2RGB)) for x in images]
            px = self.processor(images=pil, return_tensors="pt")["pixel_values"]
        chunks = [c for c in torch.tensor_split(px, len(self.devices)) if len(c)]
        def run(i, chunk):
            device, model = self.devices[i], self.models[i]
            dtype = torch.float16 if device != "cpu" else torch.float32
            if self.gpu_preprocess and device != "cpu":
                x = chunk.to(device=device, non_blocking=True)
                x = x[:, [2,1,0]].float().mul_(1.0/255.0)
                if tuple(x.shape[-2:]) != self.input_hw:
                    x = F.interpolate(
                        x, size=self.input_hw, mode="bicubic",
                        align_corners=False, antialias=True,
                    )
                mean, std = self.norm[device]
                x = ((x-mean)/std).to(dtype=dtype)
            else:
                x = chunk.to(device=device, dtype=dtype, non_blocking=True)
            with torch.inference_mode(), torch.autocast(
                device_type="cuda" if device != "cpu" else "cpu",
                dtype=torch.float16, enabled=device != "cpu"
            ):
                feat = model(x)
            if not torch.is_tensor(feat):
                raise TypeError(f"SigLIP wrapper phải trả Tensor, nhận {type(feat)!r}")
            return F.normalize(feat.float(), dim=-1).cpu().numpy()
        with ThreadPoolExecutor(max_workers=len(chunks)) as pool:
            futures = [pool.submit(run, i, chunk) for i, chunk in enumerate(chunks)]
            return np.concatenate([f.result() for f in futures], axis=0)

class PEEncoder(MultiDeviceEncoder):
    def __init__(self, pe_config, batch_per_gpu=1):
        super().__init__(batch_per_gpu)
        import core.vision_encoder.pe as pe
        import core.vision_encoder.transforms as pe_transforms
        self.kind = f"pe:{pe_config}"
        self.models = []
        self.transform = None
        for device in self.devices:
            model = pe.CLIP.from_config(pe_config, pretrained=True).eval()
            if self.transform is None:
                self.transform = pe_transforms.get_image_transform(model.image_size)
                self.image_size = int(model.image_size)
            dtype = torch.float16 if device != "cpu" else torch.float32
            self.models.append(model.to(device=device, dtype=dtype))
        probe = self.encode([np.zeros((32,32,3), np.uint8)])
        self.dim = probe.shape[1]

    def _encode_batch(self, images):
        ts = torch.stack([
            self.transform(Image.fromarray(cv2.cvtColor(x, cv2.COLOR_BGR2RGB))) for x in images
        ])
        chunks = [c for c in torch.tensor_split(ts, len(self.devices)) if len(c)]
        def run(i, chunk):
            device, model = self.devices[i], self.models[i]
            dtype = torch.float16 if device != "cpu" else torch.float32
            with torch.inference_mode(), torch.autocast(
                device_type="cuda" if device != "cpu" else "cpu",
                dtype=torch.float16, enabled=device != "cpu"
            ):
                feat = model.encode_image(chunk.to(device=device, dtype=dtype, non_blocking=True))
            return F.normalize(feat.float(), dim=-1).cpu().numpy()
        with ThreadPoolExecutor(max_workers=len(chunks)) as pool:
            futures = [pool.submit(run, i, chunk) for i, chunk in enumerate(chunks)]
            return np.concatenate([f.result() for f in futures], axis=0)

def build_selector():
    cfg = CONFIG["selector"]
    backend = cfg["backend"].lower()
    if backend == "resnet50":
        return ResNet50Encoder(cfg["batch_per_gpu"], cfg["allow_untrained"])
    if backend == "siglip":
        return SigLIPEncoder(
            cfg["model_name"], cfg["batch_per_gpu"],
            cfg.get("min_batch_per_gpu", 8),
            cfg.get("auto_reduce_batch_on_oom", True),
        )
    if backend == "pe":
        return PEEncoder(cfg["pe_config"], cfg["batch_per_gpu"])
    raise ValueError(f"Selector backend không hỗ trợ: {backend}")

def build_final_encoder(selector):
    cfg = CONFIG["final_embeddings"]
    if CONFIG["run_mode"] == "batch" and CONFIG["storage"]["batch_registry_only"]:
        log("Batch registry-only: không load final PE encoder.")
        return None
    backend = cfg["backend"].lower()
    if backend == "none": return None
    if backend == "selector": return selector
    if backend == "pe":
        if isinstance(selector, PEEncoder) and selector.kind == f"pe:{cfg['pe_config']}":
            return selector
        try:
            return PEEncoder(cfg["pe_config"], cfg["batch_per_gpu"])
        except Exception as exc:
            if cfg["required"]: raise
            log(f"WARNING: PE final encoder không load được ({type(exc).__name__}: {exc}). "
                "Sẽ chỉ lưu selector embeddings.")
            return None
    raise ValueError(f"Final embedding backend không hỗ trợ: {backend}")


## 7. Pass 1 — fused native sentinel + semantic probe 10 FPS

Cache key chứa metadata video và cấu hình/model. Cache cũ không khớp sẽ tự bị bỏ qua. Sentinel lưu raw signal; z-score được tính ở bước sau để có thể tune threshold không cần decode lại.

Backend `torch_cuda` gom native frame thành batch lớn và chuyển grayscale, Sobel/text
edge, temporal difference, grid/top-tile reduction lên GPU 0. PyAV decode ở CPU chạy
song song bằng producer–consumer queue; SigLIP đồng thời chia batch qua cả GPU 0/1.
Farneback motion 5 FPS vẫn giữ trên CPU và overlap với CUDA kernels. Nếu CUDA native
gặp lỗi, pipeline báo warning rồi fallback sang implementation OpenCV CPU tương đương.


In [ ]:
def pass1_signature(meta: VideoMeta, max_seconds=None):
    return fingerprint({
        "pipeline": CONFIG.get("pass1_version", CONFIG["version"]), "path": meta.path,
        "size": Path(meta.path).stat().st_size,
        "mtime": int(Path(meta.path).stat().st_mtime),
        "fps": meta.fps, "total": meta.total_frames,
        "max_seconds": max_seconds,
        "probe_fps": CONFIG["probe_fps"],
        "sentinel_width": CONFIG["sentinel_width"],
        "grid": CONFIG["sentinel_grid"],
        "sentinel_top_tiles": CONFIG["sentinel_top_tiles"],
        "text_grid": CONFIG["text_grid"],
        "text_top_tiles": CONFIG["text_top_tiles"],
        "motion_fps": CONFIG["motion_fps"],
        "native_backend": CONFIG["performance"]["native_backend"],
        "gpu_siglip_preprocess": CONFIG["performance"]["gpu_siglip_preprocess"],
        "selector": CONFIG["selector"],
    })

def av_frame_time(frame, decode_idx: int, fps: float) -> float:
    if frame.time is not None: return float(frame.time)
    if frame.pts is not None and frame.time_base is not None:
        return float(frame.pts * frame.time_base)
    return decode_idx / max(fps, 1e-9)

def scaled_hw(width: int, height: int, target_width: int):
    if width <= 0 or height <= 0: return target_width, int(round(target_width*9/16))
    out_h = max(2, int(round(height*target_width/width)))
    if out_h % 2: out_h += 1
    return target_width, out_h

def tile_top_delta(diff: np.ndarray, rows: int, cols: int, top_k: int) -> float:
    h, w = diff.shape
    values = []
    for r in range(rows):
        y0, y1 = r*h//rows, (r+1)*h//rows
        for c in range(cols):
            x0, x1 = c*w//cols, (c+1)*w//cols
            values.append(float(diff[y0:y1, x0:x1].mean()))
    return float(np.mean(sorted(values)[-max(1, top_k):]))

def _atomic_npz(path: Path, **arrays):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(tmp, path)

def load_pass1_cache(meta: VideoMeta, signature: str):
    ppath = OUT / "cache/probe" / f"{meta.video_id}.npz"
    spath = OUT / "cache/sentinel" / f"{meta.video_id}.npz"
    if not CONFIG["cache_pass1"] or not ppath.exists() or not spath.exists():
        return None
    try:
        probe = dict(np.load(ppath, allow_pickle=False))
        sentinel = dict(np.load(spath, allow_pickle=False))
        psig = str(probe["signature"].item())
        ssig = str(sentinel["signature"].item())
        if psig == signature == ssig:
            log(meta.video_id, "reuse pass-1 cache")
            return probe, sentinel
    except Exception as exc:
        log(meta.video_id, "cache invalid:", exc)
    return None

def fused_pass1_cpu(meta: VideoMeta, selector, max_seconds=None, reuse_cache=True,
                    write_cache=True):
    signature = pass1_signature(meta, max_seconds)
    if reuse_cache:
        cached = load_pass1_cache(meta, signature)
        if cached is not None:
            return cached

    rows, cols = CONFIG["sentinel_grid"]
    top_k = CONFIG["sentinel_top_tiles"]
    text_rows, text_cols = CONFIG["text_grid"]
    text_top_k = CONFIG["text_top_tiles"]

    native_idx, native_times = [], []
    global_raw, local_raw, text_raw, text_local_raw, motion_raw = [], [], [], [], []
    probe_idx, probe_times, quality, sharpness, phashes, text_hashes, text_density = [], [], [], [], [], [], []
    feature_parts, image_batch, image_batch_meta = [], [], []
    prev_gray = prev_broadcast_text = prev_full_text = prev_motion = None
    last_motion = 0.0
    next_probe_time = next_motion_time = 0.0
    probe_step = 1.0 / CONFIG["probe_fps"]
    motion_step = 1.0 / CONFIG["motion_fps"] if CONFIG["motion_fps"] > 0 else None

    def flush_probe():
        nonlocal image_batch, image_batch_meta
        if not image_batch: return
        feature_parts.append(selector.encode(image_batch))
        probe_idx.extend(x[0] for x in image_batch_meta)
        probe_times.extend(x[1] for x in image_batch_meta)
        image_batch, image_batch_meta = [], []

    container = av.open(meta.path)
    stream = container.streams.video[0]
    stream.thread_type = "AUTO"
    target_w, target_h = scaled_hw(
        int(stream.codec_context.width or meta.width),
        int(stream.codec_context.height or meta.height),
        CONFIG["sentinel_width"],
    )
    started = time.time()
    decoded = 0
    for av_frame in container.decode(video=0):
        pts_time = av_frame_time(av_frame, decoded, meta.fps)
        if max_seconds is not None and pts_time > float(max_seconds): break
        # frame_idx là decode-order index thật; pts_time giữ timestamp PTS độc lập.
        frame_idx = decoded
        small = av_frame.reformat(width=target_w, height=target_h, format="bgr24").to_ndarray()
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        broadcast_text = broadcast_text_edge_map(gray)
        full_text = full_text_edge_map(gray)
        if prev_gray is None:
            g = l = t = tl = 0.0
        else:
            diff = cv2.absdiff(gray, prev_gray).astype(np.float32) / 255.0
            g = float(diff.mean())
            l = tile_top_delta(diff, rows, cols, top_k)
            broadcast_diff = cv2.absdiff(
                broadcast_text, prev_broadcast_text
            ).astype(np.float32) / 255.0
            full_text_diff = cv2.absdiff(
                full_text, prev_full_text
            ).astype(np.float32) / 255.0
            t = float(broadcast_diff.mean())
            tl = tile_top_delta(full_text_diff, text_rows, text_cols, text_top_k)
        if motion_step is not None and pts_time + 1e-9 >= next_motion_time:
            motion_gray = cv2.resize(gray, (160, 90), interpolation=cv2.INTER_AREA)
            if prev_motion is not None:
                flow = cv2.calcOpticalFlowFarneback(prev_motion, motion_gray, None,
                    0.5, 2, 11, 2, 5, 1.1, 0)
                last_motion = float(np.median(cv2.magnitude(flow[...,0], flow[...,1])))
            prev_motion = motion_gray
            while next_motion_time <= pts_time + 1e-9: next_motion_time += motion_step

        native_idx.append(frame_idx); native_times.append(pts_time)
        global_raw.append(g); local_raw.append(l)
        text_raw.append(t); text_local_raw.append(tl); motion_raw.append(last_motion)
        prev_gray = gray
        prev_broadcast_text, prev_full_text = broadcast_text, full_text

        if pts_time + 1e-9 >= next_probe_time:
            q, sh, ph, th, td = probe_frame_metrics(small)
            quality.append(q); sharpness.append(sh); phashes.append(ph)
            text_hashes.append(th); text_density.append(td)
            image_batch.append(small); image_batch_meta.append((frame_idx, pts_time))
            while next_probe_time <= pts_time + 1e-9: next_probe_time += probe_step
            if len(image_batch) >= selector.total_batch_size:
                flush_probe()
        decoded += 1
        if decoded % 5000 == 0:
            elapsed = max(time.time() - started, 1e-6)
            total_txt = f"/{meta.total_frames:,}" if meta.total_frames > 0 else ""
            log(meta.video_id, f"decoded {decoded:,}{total_txt} ({decoded/elapsed:.1f} fps)")
    container.close()
    flush_probe()
    if not probe_idx or not feature_parts:
        raise RuntimeError(f"Không decode/encode được probe frame: {meta.path}")

    features = np.concatenate(feature_parts, axis=0)
    features /= np.linalg.norm(features, axis=1, keepdims=True) + 1e-9
    dtype = np.float16 if CONFIG["probe_cache_dtype"] == "float16" else np.float32
    probe = {
        "signature": np.array(signature), "frame_idx": np.asarray(probe_idx, np.int32),
        "pts_time": np.asarray(probe_times, np.float64),
        "features": features.astype(dtype), "quality": np.asarray(quality, np.float16),
        "sharpness": np.asarray(sharpness, np.float32), "phash": np.asarray(phashes, np.uint64),
        "text_hash": np.asarray(text_hashes, np.uint64),
        "text_density": np.asarray(text_density, np.float16),
        "selector_kind": np.array(selector.kind),
    }
    sentinel = {
        "signature": np.array(signature), "frame_idx": np.asarray(native_idx, np.int32),
        "pts_time": np.asarray(native_times, np.float64),
        "global_raw": np.asarray(global_raw, np.float16),
        "local_raw": np.asarray(local_raw, np.float16),
        "text_raw": np.asarray(text_raw, np.float16),
        "text_local_raw": np.asarray(text_local_raw, np.float16),
        "motion_raw": np.asarray(motion_raw, np.float16),
    }
    if (write_cache and CONFIG["cache_pass1"]
            and CONFIG["storage"]["keep_pass1_cache"] and max_seconds is None):
        _atomic_npz(OUT / "cache/probe" / f"{meta.video_id}.npz", **probe)
        _atomic_npz(OUT / "cache/sentinel" / f"{meta.video_id}.npz", **sentinel)
    log(meta.video_id, f"pass1: native={len(native_idx):,}, probe={len(probe_idx):,}, dim={features.shape[1]}")
    return probe, sentinel

class TorchNativePreprocessor:
    """Batched native sentinel on CUDA; Farneback and hashes stay on CPU."""
    def __init__(self, meta: VideoMeta):
        self.meta = meta
        requested = str(CONFIG["performance"]["native_device"])
        if not torch.cuda.is_available():
            raise RuntimeError("Torch CUDA native preprocessing cần GPU")
        self.device = torch.device(requested)
        self.prev_gray = self.prev_broadcast = self.prev_full = None
        self.prev_motion = None
        self.last_motion = 0.0
        self.next_motion_time = 0.0
        self.motion_step = (
            1.0/CONFIG["motion_fps"] if CONFIG["motion_fps"] > 0 else None
        )
        self.profile = defaultdict(float)
        self._tile_cache = {}
        self.kx = torch.tensor(
            [[-1,0,1],[-2,0,2],[-1,0,1]],
            device=self.device, dtype=torch.float32,
        ).view(1,1,3,3)
        self.ky = torch.tensor(
            [[-1,-2,-1],[0,0,0],[1,2,1]],
            device=self.device, dtype=torch.float32,
        ).view(1,1,3,3)

    def _binary_edges(self, gray):
        padded = F.pad(gray, (1,1,1,1), mode="reflect")
        gx = F.conv2d(padded, self.kx)
        gy = F.conv2d(padded, self.ky)
        magnitude = torch.sqrt(gx.square() + gy.square() + 1e-12)
        flat = magnitude.flatten(1)
        # kthvalue tránh full sort của torch.quantile trên hàng chục triệu pixel/batch.
        q75 = torch.kthvalue(
            flat, k=max(1, int(math.ceil(0.75*flat.shape[1]))), dim=1
        ).values
        threshold = torch.maximum(q75, torch.full_like(q75, 24.0)).view(-1,1,1,1)
        edge = (magnitude >= threshold).float()
        # Morphological close với horizontal kernel 3×1.
        dilated = F.max_pool2d(edge, kernel_size=(1,3), stride=1, padding=(0,1))
        return -F.max_pool2d(-dilated, kernel_size=(1,3), stride=1, padding=(0,1))

    def _with_previous(self, current, previous):
        if previous is None:
            first = torch.zeros_like(current[:1])
        else:
            first = (current[:1]-previous).abs()
        if len(current) == 1:
            return first
        return torch.cat([first, (current[1:]-current[:-1]).abs()], dim=0)

    def _tile_top_mean(self, diff, rows, cols, top_k):
        _, _, h, w = diff.shape
        key = (h, w, int(rows), int(cols))
        cached = self._tile_cache.get(key)
        if cached is None:
            y0 = torch.tensor([r*h//rows for r in range(rows)], device=self.device)
            y1 = torch.tensor([(r+1)*h//rows for r in range(rows)], device=self.device)
            x0 = torch.tensor([c*w//cols for c in range(cols)], device=self.device)
            x1 = torch.tensor([(c+1)*w//cols for c in range(cols)], device=self.device)
            area = ((y1-y0)[:,None]*(x1-x0)[None,:]).float().view(1,1,rows,cols)
            cached = (y0, y1, x0, x1, area)
            self._tile_cache[key] = cached
        y0, y1, x0, x1, area = cached
        integral = F.pad(diff.cumsum(2).cumsum(3), (1,0,1,0))
        tiles = (
            integral[:,:,y1[:,None],x1[None,:]]
            - integral[:,:,y0[:,None],x1[None,:]]
            - integral[:,:,y1[:,None],x0[None,:]]
            + integral[:,:,y0[:,None],x0[None,:]]
        ) / area
        k = max(1, min(int(top_k), rows*cols))
        return torch.topk(tiles.flatten(2), k=k, dim=-1).values.mean(-1).squeeze(1)

    def _cpu_motion_and_probe_metrics(self, images, times, probe_positions):
        motion = []
        for image, pts_time in zip(images, times):
            if self.motion_step is not None and pts_time + 1e-9 >= self.next_motion_time:
                gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
                motion_gray = cv2.resize(gray, (160,90), interpolation=cv2.INTER_AREA)
                if self.prev_motion is not None:
                    flow = cv2.calcOpticalFlowFarneback(
                        self.prev_motion, motion_gray, None, 0.5, 2, 11, 2, 5, 1.1, 0
                    )
                    self.last_motion = float(
                        np.median(cv2.magnitude(flow[...,0], flow[...,1]))
                    )
                self.prev_motion = motion_gray
                while self.next_motion_time <= pts_time + 1e-9:
                    self.next_motion_time += self.motion_step
            motion.append(self.last_motion)

        quality, sharpness, phashes, text_hashes, text_density = [], [], [], [], []
        for pos in probe_positions:
            image = images[int(pos)]
            q, sharp_raw, ph, th, td = probe_frame_metrics(image)
            quality.append(q); sharpness.append(sharp_raw); phashes.append(ph)
            text_hashes.append(th); text_density.append(td)
        return {
            "motion_raw": np.asarray(motion, np.float32),
            "quality": np.asarray(quality, np.float32),
            "sharpness": np.asarray(sharpness, np.float32),
            "phash": np.asarray(phashes, np.uint64),
            "text_hash": np.asarray(text_hashes, np.uint64),
            "text_density": np.asarray(text_density, np.float32),
        }

    def process(self, images, times, frame_indices, probe_mask, selector):
        started = time.time()
        arr = np.ascontiguousarray(np.stack(images, axis=0))
        host = torch.from_numpy(arr).permute(0,3,1,2)
        if torch.cuda.is_available(): host = host.pin_memory()
        probe_positions = np.flatnonzero(np.asarray(probe_mask, dtype=bool))

        with torch.inference_mode():
            bgr = host.to(self.device, non_blocking=True).float()
            gray = (0.114*bgr[:,0:1] + 0.587*bgr[:,1:2] + 0.299*bgr[:,2:3])
            h = gray.shape[-2]
            top = max(1, int(0.20*h)); bottom = min(h-1, int(0.60*h))
            broadcast_gray = torch.cat([gray[:,:,:top], gray[:,:,bottom:]], dim=2)
            full_edges = self._binary_edges(gray)
            broadcast_edges = self._binary_edges(broadcast_gray)

            gray_diff = self._with_previous(gray, self.prev_gray) / 255.0
            broadcast_diff = self._with_previous(
                broadcast_edges, self.prev_broadcast
            )
            full_text_diff = self._with_previous(full_edges, self.prev_full)

            rows, cols = CONFIG["sentinel_grid"]
            text_rows, text_cols = CONFIG["text_grid"]
            global_raw_t = gray_diff.mean(dim=(1,2,3))
            local_raw_t = self._tile_top_mean(
                gray_diff, rows, cols, CONFIG["sentinel_top_tiles"]
            )
            text_raw_t = broadcast_diff.mean(dim=(1,2,3))
            text_local_raw_t = self._tile_top_mean(
                full_text_diff, text_rows, text_cols, CONFIG["text_top_tiles"]
            )
            self.prev_gray = gray[-1:].detach().clone()
            self.prev_broadcast = broadcast_edges[-1:].detach().clone()
            self.prev_full = full_edges[-1:].detach().clone()

        # Một CPU worker giữ state Farneback theo thứ tự batch. Trong lúc nó chạy,
        # GPU 0 sync native tensors rồi cả GPU 0/1 encode SigLIP probe song song.
        probe_images = [images[int(pos)] for pos in probe_positions]
        with ThreadPoolExecutor(max_workers=1, thread_name_prefix="motion-hash") as cpu_pool:
            metrics_future = cpu_pool.submit(
                self._cpu_motion_and_probe_metrics,
                images, times, probe_positions,
            )
            native = {
                "global_raw": global_raw_t.float().cpu().numpy(),
                "local_raw": local_raw_t.float().cpu().numpy(),
                "text_raw": text_raw_t.float().cpu().numpy(),
                "text_local_raw": text_local_raw_t.float().cpu().numpy(),
            }
            # Nhả intermediate native trước khi SigLIP cần activation memory trên GPU 0.
            del (bgr, gray, broadcast_gray, full_edges, broadcast_edges,
                 gray_diff, broadcast_diff, full_text_diff,
                 global_raw_t, local_raw_t, text_raw_t, text_local_raw_t)
            encode_started = time.time()
            features = selector.encode(probe_images)
            self.profile["siglip_sec"] += time.time()-encode_started
            cpu_metrics = metrics_future.result()
        self.profile["overlapped_batch_sec"] += time.time()-started
        return {
            "native_idx": np.asarray(frame_indices, np.int32),
            "native_times": np.asarray(times, np.float64),
            **native,
            "motion_raw": cpu_metrics.pop("motion_raw"),
            "probe_idx": np.asarray(frame_indices, np.int32)[probe_positions],
            "probe_times": np.asarray(times, np.float64)[probe_positions],
            "features": features,
            **cpu_metrics,
        }

def fused_pass1_gpu(meta: VideoMeta, selector, max_seconds=None, reuse_cache=True,
                    batch_frames_override=None):
    signature = pass1_signature(meta, max_seconds)
    if reuse_cache:
        cached = load_pass1_cache(meta, signature)
        if cached is not None:
            return cached

    configured_batch = (
        CONFIG["performance"]["native_batch_frames"]
        if batch_frames_override is None else batch_frames_override
    )
    batch_frames = max(32, int(configured_batch))
    prefetch = max(1, int(CONFIG["performance"]["decode_prefetch_batches"]))
    worker = TorchNativePreprocessor(meta)
    results, pending = [], []
    images, times, indices, probe_mask = [], [], [], []
    next_probe_time = 0.0
    probe_step = 1.0/CONFIG["probe_fps"]
    started = time.time(); decoded = 0

    def consume_one():
        results.append(pending.pop(0).result())

    container = av.open(meta.path)
    try:
        stream = container.streams.video[0]; stream.thread_type = "AUTO"
        target_w, target_h = scaled_hw(
            int(stream.codec_context.width or meta.width),
            int(stream.codec_context.height or meta.height),
            CONFIG["sentinel_width"],
        )
        log(meta.video_id, f"GPU pass1 native_batch={batch_frames}, "
            f"selector_batch={selector.total_batch_size}, prefetch={prefetch}")
        with ThreadPoolExecutor(max_workers=1, thread_name_prefix="native-gpu") as pool:
            for av_frame in container.decode(video=0):
                pts_time = av_frame_time(av_frame, decoded, meta.fps)
                if max_seconds is not None and pts_time > float(max_seconds): break
                is_probe = pts_time + 1e-9 >= next_probe_time
                if is_probe:
                    while next_probe_time <= pts_time + 1e-9: next_probe_time += probe_step
                small = av_frame.reformat(
                    width=target_w, height=target_h, format="bgr24"
                ).to_ndarray()
                images.append(small); times.append(float(pts_time))
                indices.append(int(decoded)); probe_mask.append(bool(is_probe))
                decoded += 1
                if len(images) >= batch_frames:
                    pending.append(pool.submit(
                        worker.process, images, times, indices, probe_mask, selector
                    ))
                    images, times, indices, probe_mask = [], [], [], []
                    if len(pending) >= prefetch: consume_one()
                if decoded % 5000 == 0:
                    elapsed = max(time.time()-started, 1e-6)
                    total_txt = f"/{meta.total_frames:,}" if meta.total_frames > 0 else ""
                    log(meta.video_id, f"decoded {decoded:,}{total_txt} ({decoded/elapsed:.1f} fps, GPU queue={len(pending)})")
            if images:
                pending.append(pool.submit(
                    worker.process, images, times, indices, probe_mask, selector
                ))
            while pending: consume_one()
    finally:
        container.close()

    if not results:
        raise RuntimeError(f"Không decode được native batch: {meta.path}")
    cat = lambda key, dtype=None: np.concatenate([x[key] for x in results]).astype(
        dtype, copy=False
    ) if dtype is not None else np.concatenate([x[key] for x in results])
    features = cat("features", np.float32)
    if not len(features):
        raise RuntimeError(f"Không encode được probe frame: {meta.path}")
    features /= np.linalg.norm(features, axis=1, keepdims=True) + 1e-9
    cache_dtype = np.float16 if CONFIG["probe_cache_dtype"] == "float16" else np.float32
    probe = {
        "signature": np.array(signature), "frame_idx": cat("probe_idx", np.int32),
        "pts_time": cat("probe_times", np.float64),
        "features": features.astype(cache_dtype), "quality": cat("quality", np.float16),
        "sharpness": cat("sharpness", np.float32), "phash": cat("phash", np.uint64),
        "text_hash": cat("text_hash", np.uint64),
        "text_density": cat("text_density", np.float16),
        "selector_kind": np.array(selector.kind),
    }
    sentinel = {
        "signature": np.array(signature), "frame_idx": cat("native_idx", np.int32),
        "pts_time": cat("native_times", np.float64),
        "global_raw": cat("global_raw", np.float16),
        "local_raw": cat("local_raw", np.float16),
        "text_raw": cat("text_raw", np.float16),
        "text_local_raw": cat("text_local_raw", np.float16),
        "motion_raw": cat("motion_raw", np.float16),
    }
    if CONFIG["cache_pass1"] and CONFIG["storage"]["keep_pass1_cache"] and max_seconds is None:
        _atomic_npz(OUT/"cache/probe"/f"{meta.video_id}.npz", **probe)
        _atomic_npz(OUT/"cache/sentinel"/f"{meta.video_id}.npz", **sentinel)
    elapsed = time.time()-started
    log(meta.video_id, f"GPU pass1: native={len(sentinel['frame_idx']):,}, probe={len(probe['frame_idx']):,}, "
        f"throughput={len(sentinel['frame_idx'])/max(elapsed,1e-6):.1f} fps, "
        f"overlapped_batches={worker.profile['overlapped_batch_sec']:.1f}s, "
        f"SigLIP={worker.profile['siglip_sec']:.1f}s")
    return probe, sentinel

def fused_pass1(meta: VideoMeta, selector, max_seconds=None, reuse_cache=True):
    backend = str(CONFIG["performance"]["native_backend"]).lower()
    use_gpu = backend == "torch_cuda" and torch.cuda.is_available()
    if use_gpu:
        native_batch = max(32, int(CONFIG["performance"]["native_batch_frames"]))
        min_native_batch = max(
            32, int(CONFIG["performance"].get("min_native_batch_frames", 32))
        )
        try:
            while True:
                try:
                    return fused_pass1_gpu(
                        meta, selector, max_seconds, reuse_cache,
                        batch_frames_override=native_batch,
                    )
                except RuntimeError as exc:
                    is_oom = "out of memory" in str(exc).lower()
                    next_batch = max(min_native_batch, native_batch//2)
                    can_retry = (
                        is_oom
                        and CONFIG["performance"].get(
                            "auto_reduce_native_batch_on_oom", True
                        )
                        and next_batch < native_batch
                    )
                    if not can_retry: raise
                    log(meta.video_id, f"CUDA native OOM: giảm batch "
                        f"{native_batch} → {next_batch} và decode lại video")
                    native_batch = next_batch
                    gc.collect(); torch.cuda.empty_cache()
        except Exception as exc:
            if not CONFIG["performance"].get("fallback_to_cpu", True): raise
            log(meta.video_id, f"WARNING: GPU native pass failed ({type(exc).__name__}: {exc}); fallback CPU")
            traceback.print_exc()
            gc.collect(); torch.cuda.empty_cache()
            return fused_pass1_cpu(
                meta, selector, max_seconds, reuse_cache=False, write_cache=False
            )
    if backend == "torch_cuda":
        log(meta.video_id, "CUDA không khả dụng; dùng OpenCV CPU pass1")
    return fused_pass1_cpu(meta, selector, max_seconds, reuse_cache)


## 8. Native sentinel events — rolling median/MAD + temporal NMS


In [ ]:
def rolling_robust_z(values: np.ndarray, window: int):
    x = np.asarray(values, np.float32)
    window = max(3, int(window) | 1)
    med = median_filter(x, size=window, mode="nearest")
    mad = median_filter(np.abs(x - med), size=window, mode="nearest")
    # floor nhỏ chống z-score nổ khi cảnh gần như tĩnh tuyệt đối
    scale_floor = max(1e-4, float(np.median(mad)) * 0.25)
    return (x - med) / (1.4826 * np.maximum(mad, scale_floor) + 1e-4)

def peak_nms(indices, raw, z, z_threshold, raw_floor, radius_frames):
    indices = np.asarray(indices, np.int64)
    raw, z = np.asarray(raw, np.float32), np.asarray(z, np.float32)
    eligible = np.flatnonzero((z >= z_threshold) & (raw >= raw_floor))
    order = eligible[np.argsort(z[eligible])[::-1]]
    kept = []
    for pos in order:
        frame = int(indices[pos])
        if all(abs(frame - int(indices[q])) > radius_frames for q in kept):
            kept.append(int(pos))
    return sorted(kept, key=lambda p: int(indices[p]))

def merge_events(events: list[Event], radius_frames: int):
    if not events: return []
    events = sorted(events, key=lambda e: (e.pts_time if e.pts_time is not None else e.frame_idx, e.priority))
    merged = [events[0]]
    for event in events[1:]:
        prev = merged[-1]
        if abs(event.frame_idx - prev.frame_idx) <= radius_frames:
            if max(event.scores.values() or [0]) > max(prev.scores.values() or [0]):
                prev.frame_idx = event.frame_idx
                prev.pts_time = event.pts_time
            prev.start_frame = min(prev.start_frame, event.start_frame)
            prev.end_frame = max(prev.end_frame, event.end_frame)
            if event.start_time is not None:
                prev.start_time = event.start_time if prev.start_time is None else min(prev.start_time, event.start_time)
            if event.end_time is not None:
                prev.end_time = event.end_time if prev.end_time is None else max(prev.end_time, event.end_time)
            prev.radius_ms = max(prev.radius_ms or 0, event.radius_ms or 0) or None
            prev.sources |= event.sources
            prev.priority = min(prev.priority, event.priority)
            for k, v in event.scores.items(): prev.scores[k] = max(prev.scores.get(k, -np.inf), v)
        else:
            merged.append(event)
    return merged

def sentinel_events(meta: VideoMeta, sentinel: dict):
    cfg = CONFIG["sentinel"]
    idx = sentinel["frame_idx"].astype(np.int64)
    times = sentinel["pts_time"].astype(np.float64)
    window = int(round(CONFIG["rolling_window_sec"] * meta.fps))
    radius = max(1, int(round(CONFIG["infoshoot"]["temporal_nms_ms"] * meta.fps / 1000)))
    specs = [
        ("native_global", "global_raw", cfg["global_z"], cfg["global_min_raw"]),
        ("native_local", "local_raw", cfg["local_z"], cfg["local_min_raw"]),
        ("text_event", "text_raw", cfg["text_z"], cfg["text_min_raw"]),
        ("text_local_event", "text_local_raw", cfg["text_local_z"], cfg["text_local_min_raw"]),
        ("motion_event", "motion_raw", cfg["motion_z"], cfg["motion_min_raw"]),
    ]
    events, z_arrays = [], {}
    for source, key, z_thr, raw_floor in specs:
        raw = sentinel[key].astype(np.float32)
        z = rolling_robust_z(raw, window)
        z_arrays[source] = z
        for pos in peak_nms(idx, raw, z, z_thr, raw_floor, radius):
            value = float(z[pos])
            priority = 0 if source in {"text_event", "text_local_event"} or value >= cfg["extreme_z"] else 2
            events.append(Event(
                frame_idx=int(idx[pos]), start_frame=int(idx[pos]), end_frame=int(idx[pos]),
                pts_time=float(times[pos]), start_time=float(times[pos]), end_time=float(times[pos]),
                radius_ms=CONFIG["microburst"]["radius_ms"],
                sources={source}, priority=priority,
                scores={f"{source}_z": value, f"{source}_raw": float(raw[pos])},
            ))
    return merge_events(events, radius), z_arrays


## 9. InfoShot GLRT, content-driven segments, COMMON và MULTI-UNIQUE

GLRT được tính bằng cumulative feature sums, không tạo affinity matrix `T×T`. Với normalized embedding, mean pairwise cosine trong window có thể suy ra từ norm của tổng vector; kết quả tương đương nhưng giảm memory mạnh.


In [ ]:
def glrt_boundary_scores(features: np.ndarray, window_min=3, window_max=15):
    e = np.asarray(features, np.float32)
    e /= np.linalg.norm(e, axis=1, keepdims=True) + 1e-9
    n, dim = e.shape
    if (CONFIG["performance"].get("gpu_semantic_glrt", True)
            and torch.cuda.is_available()):
        device = torch.device(CONFIG["performance"].get("native_device", "cuda:0"))
        with torch.inference_mode():
            et = torch.from_numpy(e).to(device=device, non_blocking=True)
            cumulative = torch.cat([
                torch.zeros((1, dim), device=device, dtype=torch.float32),
                torch.cumsum(et, dim=0),
            ], dim=0)
            best = torch.full((n,), -torch.inf, device=device, dtype=torch.float32)
            best_window = torch.zeros((n,), device=device, dtype=torch.int16)
            for k in range(int(window_min), int(window_max) + 1):
                if n < 2*k: break
                t = torch.arange(k, n-k+1, device=device)
                left_sum = cumulative[t] - cumulative[t-k]
                right_sum = cumulative[t+k] - cumulative[t]
                denom = max(1, k*(k-1))
                left_within = (left_sum.square().sum(-1)-k)/denom
                right_within = (right_sum.square().sum(-1)-k)/denom
                cross = (left_sum*right_sum).sum(-1)/(k*k)
                score = 0.5*(left_within+right_within)-cross
                improve = score > best[t]
                selected = t[improve]
                best[selected] = score[improve]
                best_window[selected] = k
            best = torch.where(torch.isfinite(best), best, torch.zeros_like(best))
            return best.cpu().numpy(), best_window.cpu().numpy()

    cumulative = np.vstack([np.zeros((1, dim), np.float32), np.cumsum(e, axis=0)])
    best = np.full(n, -np.inf, np.float32)
    best_window = np.zeros(n, np.int16)
    for k in range(int(window_min), int(window_max) + 1):
        if n < 2*k: break
        t = np.arange(k, n-k+1)
        left_sum = cumulative[t] - cumulative[t-k]
        right_sum = cumulative[t+k] - cumulative[t]
        # mean cosine excluding diagonal within each side
        denom = k * (k - 1)
        left_within = (np.einsum("ij,ij->i", left_sum, left_sum) - k) / max(1, denom)
        right_within = (np.einsum("ij,ij->i", right_sum, right_sum) - k) / max(1, denom)
        cross = np.einsum("ij,ij->i", left_sum, right_sum) / (k*k)
        score = 0.5*(left_within + right_within) - cross
        improve = score > best[t]
        best[t[improve]] = score[improve]
        best_window[t[improve]] = k
    best[~np.isfinite(best)] = 0.0
    return best, best_window

def _enforce_boundary_spacing(boundaries, scores, n, min_len):
    """
    Chỉ nhận boundary nội bộ tạo được ít nhất ``min_len`` probe frames
    ở cả hai phía. Boundary đầu/cuối, quá sát mép, trùng hoặc malformed
    bị bỏ qua thay vì làm ``max()/min()`` nhận iterable rỗng.
    """
    n = int(n)
    min_len = max(1, int(min_len))
    if n < 2*min_len:
        return []

    candidates = set()
    if boundaries is not None:
        for value in boundaries:
            try:
                t = int(value)
            except (TypeError, ValueError, OverflowError):
                continue
            if min_len <= t <= n-min_len and 0 <= t < len(scores):
                candidates.add(t)

    chosen = []
    for t in sorted(candidates, key=lambda x: float(scores[x]), reverse=True):
        edges = sorted([0, *chosen, n])
        left_candidates = [x for x in edges if x < t]
        right_candidates = [x for x in edges if x > t]
        if not left_candidates or not right_candidates:
            continue
        left = max(left_candidates)
        right = min(right_candidates)
        if t-left >= min_len and right-t >= min_len:
            chosen.append(t)
    return sorted(chosen)

def content_driven_segments(boundary_score, boundary_z, forced_boundaries=None):
    cfg = CONFIG["segment"]
    n = len(boundary_score)
    min_len, max_len = cfg["min_probe_frames"], cfg["max_probe_frames"]
    local_peaks = [
        i for i in range(1, n-1)
        if boundary_z[i] >= cfg["boundary_z"]
        and boundary_score[i] >= cfg["boundary_min_raw"]
        and boundary_score[i] >= boundary_score[i-1]
        and boundary_score[i] >= boundary_score[i+1]
    ]
    seeds = list(local_peaks) + list(forced_boundaries or [])
    chosen = _enforce_boundary_spacing(seeds, boundary_score, n, min_len)

    # Bắt buộc chia analysis segment >30s tại boundary mạnh nhất khả dụng.
    while True:
        edges = [0, *chosen, n]
        long_pair = next(((a,b) for a,b in zip(edges[:-1], edges[1:]) if b-a > max_len), None)
        if long_pair is None: break
        a, b = long_pair
        lo, hi = a + min_len, b - min_len
        target_hi = min(hi, a + max_len)
        if target_hi <= lo:
            split = min(b-min_len, a+max_len)
        else:
            local = boundary_score[lo:target_hi+1]
            split = lo + int(np.argmax(local)) if len(local) else min(b-min_len, a+max_len)
        chosen.append(int(split)); chosen = sorted(set(chosen))
    edges = [0, *chosen, n]
    return [(sid, a, b-1) for sid, (a,b) in enumerate(zip(edges[:-1], edges[1:])) if b > a]

def nearest_probe_pos(probe_frames: np.ndarray, native_frame: int) -> int:
    pos = int(np.searchsorted(probe_frames, native_frame))
    options = [p for p in (pos-1, pos) if 0 <= p < len(probe_frames)]
    return min(options, key=lambda p: abs(int(probe_frames[p]) - int(native_frame)))

def segment_for_probe_pos(segments, pos):
    for sid, a, b in segments:
        if a <= pos <= b: return sid
    return -1

def candidate_from_probe(meta, probe, pos, segment_id, sources, priority, scores=None):
    fi = int(probe["frame_idx"][pos])
    return FrameCandidate(
        frame_idx=fi, pts_time=float(probe["pts_time"][pos]), segment_id=int(segment_id),
        sources=set(sources), priority=int(priority), scores=dict(scores or {}),
        quality=float(probe["quality"][pos]), phash=int(probe["phash"][pos]),
        text_hash=int(probe["text_hash"][pos]), text_density=float(probe["text_density"][pos]),
        embedding=np.asarray(probe["features"][pos], np.float32),
    )

def infoshoot_candidates(meta: VideoMeta, probe: dict, segments):
    e = np.asarray(probe["features"], np.float32)
    e /= np.linalg.norm(e, axis=1, keepdims=True) + 1e-9
    cfg = CONFIG["infoshoot"]
    candidates, semantic_events, short_events = [], [], []
    for sid, a, b in segments:
        idx = np.arange(a, b+1)
        seg = e[idx]
        length = len(seg)
        if length == 1:
            candidates.append(candidate_from_probe(meta, probe, a, sid, {"common"}, 1))
            continue
        sums = seg.sum(axis=0)
        typicality = (seg @ sums - 1.0) / max(1, length-1)
        volatility = np.zeros(length, np.float32)
        k = int(cfg["neighborhood_k"])
        for local_i in range(length):
            neighbors = [j for j in range(max(0,local_i-k), min(length,local_i+k+1)) if j != local_i]
            volatility[local_i] = 1.0 - float(np.mean(seg[neighbors] @ seg[local_i])) if neighbors else 0.0
        g, v = minmax(typicality), minmax(volatility)
        common_score = cfg["lambda_common"]*g - (1-cfg["lambda_common"])*v
        q = np.asarray(probe["quality"][idx], np.float32)
        usable = q >= max(0.25, float(np.quantile(q, 0.20)))
        ranked = np.argsort(common_score + 0.15*q)[::-1]
        local_common = next((int(x) for x in ranked if usable[x]), int(ranked[0]))
        common_pos = a + local_common
        candidates.append(candidate_from_probe(meta, probe, common_pos, sid, {"common"}, 1, {
            "typicality": float(g[local_common]), "volatility": float(v[local_common]),
            "common_score": float(common_score[local_common]),
        }))
        common_fi = int(probe["frame_idx"][common_pos]); common_t = float(probe["pts_time"][common_pos])
        semantic_events.append(Event(
            frame_idx=common_fi, start_frame=common_fi, end_frame=common_fi,
            pts_time=common_t, start_time=common_t, end_time=common_t,
            radius_ms=CONFIG["microburst"]["common_radius_ms"],
            sources={"common"}, priority=1,
            scores={"common_score": float(common_score[local_common])}, segment_id=sid,
        ))

        unique_score = cfg["alpha_unique"]*(1-g) + (1-cfg["alpha_unique"])*v
        uz = robust_z_global(unique_score)
        radius_probe = max(1, int(round(cfg["temporal_nms_ms"]*CONFIG["probe_fps"]/1000)))
        peaks = [
            i for i in range(length)
            if unique_score[i] >= cfg["unique_min_score"] and uz[i] >= cfg["unique_z"]
            and i != local_common
            and unique_score[i] >= unique_score[max(0,i-1)]
            and unique_score[i] >= unique_score[min(length-1,i+1)]
        ]
        selected = []
        primary = None
        if length >= 3 and cfg.get("keep_primary_unique", True):
            eligible_primary = [i for i in range(length) if i != local_common]
            primary = max(eligible_primary, key=lambda i: float(unique_score[i]))
            selected.append(primary)
        for local_i in sorted(peaks, key=lambda x: float(unique_score[x]), reverse=True):
            if all(abs(local_i-j) > radius_probe for j in selected):
                selected.append(local_i)
        for local_i in sorted(selected):
            pos = a + local_i
            scores = {"typicality": float(g[local_i]), "volatility": float(v[local_i]),
                      "unique_score": float(unique_score[local_i]), "unique_z": float(uz[local_i])}
            reason = "semantic_unique_primary" if local_i == primary else "semantic_unique"
            candidates.append(candidate_from_probe(meta, probe, pos, sid, {reason}, 1, scores))
            fi = int(probe["frame_idx"][pos])
            pt = float(probe["pts_time"][pos])
            semantic_events.append(Event(
                frame_idx=fi, start_frame=fi, end_frame=fi,
                pts_time=pt, start_time=pt, end_time=pt,
                radius_ms=CONFIG["microburst"]["radius_ms"],
                sources={reason}, priority=1, scores=scores, segment_id=sid,
            ))

        start_f, end_f = int(probe["frame_idx"][a]), int(probe["frame_idx"][b])
        start_t, end_t = float(probe["pts_time"][a]), float(probe["pts_time"][b])
        duration_ms = max(0.0, end_t-start_t)*1000
        if duration_ms <= CONFIG["microburst"]["short_segment_ms"]:
            mid_pos = (a+b)//2
            short_events.append(Event(
                frame_idx=int(probe["frame_idx"][mid_pos]), start_frame=start_f, end_frame=end_f,
                pts_time=float(probe["pts_time"][mid_pos]), start_time=start_t, end_time=end_t,
                radius_ms=None,
                sources={"short_shot"}, priority=0,
                scores={"short_duration_ms": float(duration_ms)}, segment_id=sid,
            ))
    return candidates, semantic_events, short_events


## 10. Optional TransNetV2 + hợp nhất structural boundary

Tắt mặc định. Nếu bật, TransNet chỉ là auxiliary signal; GLRT/sentinel vẫn là nền chính.


In [ ]:
_TRANSNET_MODEL = None

def transnet_native_boundaries(meta: VideoMeta):
    global _TRANSNET_MODEL
    if not CONFIG["segment"]["use_transnetv2"]:
        return []
    try:
        if _TRANSNET_MODEL is None:
            from transnetv2_pytorch import TransNetV2
            _TRANSNET_MODEL = TransNetV2().eval()
            if torch.cuda.is_available(): _TRANSNET_MODEL = _TRANSNET_MODEL.to("cuda:0")
        with torch.inference_mode():
            _, single, _ = _TRANSNET_MODEL.predict_video(meta.path)
        pred = single.detach().float().cpu().numpy().reshape(-1) if hasattr(single, "detach") else np.asarray(single).reshape(-1)
        hot = np.flatnonzero(pred >= CONFIG["segment"]["transnet_threshold"])
        if not len(hot): return []
        groups, current = [], [int(hot[0])]
        for x in hot[1:]:
            x = int(x)
            if x == current[-1] + 1: current.append(x)
            else: groups.append(current); current = [x]
        groups.append(current)
        return [int(round(np.mean(g))) for g in groups]
    except Exception as exc:
        log(f"TransNetV2 fallback off ({type(exc).__name__}: {exc})")
        return []

def build_segments(meta: VideoMeta, probe: dict, sentinel: dict, sent_events: list[Event]):
    features = np.asarray(probe["features"], np.float32)
    score, best_window = glrt_boundary_scores(
        features, CONFIG["segment"]["window_min"], CONFIG["segment"]["window_max"]
    )
    bz = robust_z_global(score)
    pframes = probe["frame_idx"].astype(np.int64)
    forced = []
    for event in sent_events:
        if "native_global" in event.sources and event.scores.get("native_global_z", 0) >= CONFIG["segment"]["force_global_z"]:
            forced.append(nearest_probe_pos(pframes, event.frame_idx))
    for native in transnet_native_boundaries(meta):
        forced.append(nearest_probe_pos(pframes, native))
    segments = content_driven_segments(score, bz, forced)
    boundary_events = []
    for sid, a, _ in segments[1:]:
        if score[a] < CONFIG["segment"]["boundary_min_raw"]:
            continue
        fi, pt = int(pframes[a]), float(probe["pts_time"][a])
        boundary_events.append(Event(
            frame_idx=fi, start_frame=fi, end_frame=fi,
            pts_time=pt, start_time=pt, end_time=pt,
            radius_ms=CONFIG["microburst"]["boundary_radius_ms"],
            sources={"semantic_boundary"}, priority=1,
            scores={"boundary": float(score[a]), "boundary_z": float(bz[a])},
            segment_id=sid,
        ))
    return segments, score, bz, best_window, boundary_events


## 11. BTC safety union

Notebook chỉ union BTC khi tìm được CSV mapping có `frame_idx`, hoặc có `pts_time` để suy ra frame. Chỉ có JPEG đánh số nhưng không có mapping thì không đoán timestamp. Điều này tránh tạo identity sai.


In [ ]:
_BTC_CSV_INDEX = None

def build_btc_csv_index():
    global _BTC_CSV_INDEX
    if _BTC_CSV_INDEX is not None: return _BTC_CSV_INDEX
    index = defaultdict(list)
    if CONFIG["btc_union"]["enabled"]:
        for root, dirs, files in os.walk(DATA_ROOT):
            # Standard dataset để JPEG dưới .../keyframes/<video_id>/; mapping thường ở map-keyframes.
            # Prune nhánh JPEG để index CSV không phải enumerate toàn bộ ảnh.
            dirs[:] = [d for d in dirs if d.casefold() != "keyframes"]
            for filename in files:
                if filename.lower().endswith(".csv"):
                    path = Path(root) / filename
                    index[canonical_video_id(path.stem)].append(path)
    _BTC_CSV_INDEX = dict(index)
    log(f"BTC mapping index: {sum(map(len,index.values())):,} CSV, {len(index):,} video IDs")
    return _BTC_CSV_INDEX

def _find_column(df, choices):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for name in choices:
        if name in lookup: return lookup[name]
    return None

def load_btc_candidates(meta: VideoMeta, probe: dict, segments):
    if not CONFIG["btc_union"]["enabled"]: return []
    paths = build_btc_csv_index().get(meta.video_id, [])
    chosen_df, chosen_path = None, None
    for path in paths:
        try:
            df = pd.read_csv(path)
            frame_col = _find_column(df, ["frame_idx", "frameid", "frame_id", "frame"])
            time_col = _find_column(df, ["pts_time", "timestamp_sec", "time_sec", "seconds"])
            if frame_col is not None or time_col is not None:
                chosen_df, chosen_path = df, path; break
        except Exception: pass
    if chosen_df is None:
        if CONFIG["btc_union"]["require_mapping"]:
            raise RuntimeError(f"Không tìm thấy BTC mapping có frame_idx/pts_time cho {meta.video_id}")
        return []
    frame_col = _find_column(chosen_df, ["frame_idx", "frameid", "frame_id", "frame"])
    time_col = _find_column(chosen_df, ["pts_time", "timestamp_sec", "time_sec", "seconds"])
    n_col = _find_column(chosen_df, ["n", "keyframe_id", "keyframe_no", "index"])
    out, pframes = [], probe["frame_idx"].astype(np.int64)
    for row_i, row in chosen_df.iterrows():
        try:
            fi = int(round(float(row[frame_col]))) if frame_col is not None else int(round(float(row[time_col])*meta.fps))
        except Exception:
            continue
        if not (0 <= fi < meta.total_frames): continue
        pos = nearest_probe_pos(pframes, fi)
        sid = segment_for_probe_pos(segments, pos)
        n = row[n_col] if n_col is not None else row_i + 1
        btc_id = f"{meta.category}/{meta.video_id}/{int(n):03d}" if str(n).replace(".", "", 1).isdigit() else str(n)
        c = candidate_from_probe(meta, probe, pos, sid, {"btc"}, 0, {"btc_mapping": 1.0})
        try:
            exact_time = float(row[time_col]) if time_col is not None else fi/meta.fps
        except Exception:
            exact_time = fi/meta.fps
        c.frame_idx, c.pts_time, c.btc_id = fi, exact_time, btc_id
        out.append(c)
    log(meta.video_id, f"BTC union {len(out)} frame từ {chosen_path}")
    return out


## 12. Native-FPS micro-burst — PyAV sequential pass

Decode tuần tự toàn video lần hai theo PTS; các cửa sổ đang active dùng chung frame low-res nên không random-seek và không giữ RGB full-resolution trong RAM. Refinement tính cả full-frame text mean và localized text grid; `text_local_event` ưu tiên localized evidence. Frame thứ hai chỉ giữ khi đủ score và bổ sung visual/text state. Candidate refine giữ nguyên priority của event gốc.


In [ ]:
def _burst_metrics(frames: dict[int, tuple[float, np.ndarray]]):
    ids = sorted(frames)
    images = [frames[i][1] for i in ids]
    grays = [cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) for image in images]
    base_shape = grays[0].shape[::-1]
    grays = [cv2.resize(g, base_shape, interpolation=cv2.INTER_AREA) for g in grays]
    background = np.median(np.stack(grays).astype(np.float32), axis=0).astype(np.uint8)
    base_text = full_text_edge_map(background)
    rows, cols = CONFIG["sentinel_grid"]
    text_rows, text_cols = CONFIG["text_grid"]
    metrics = []
    for fi, frame, gray in zip(ids, images, grays):
        diff = cv2.absdiff(gray, background).astype(np.float32)/255.0
        global_d = float(diff.mean())
        local_d = tile_top_delta(diff, rows, cols, CONFIG["sentinel_top_tiles"])
        edges = full_text_edge_map(gray)
        text_diff = cv2.absdiff(edges, base_text).astype(np.float32)/255.0
        text_global_d = float(text_diff.mean())
        text_local_d = tile_top_delta(
            text_diff, text_rows, text_cols, CONFIG["text_top_tiles"]
        )
        q, sharp, mean, clipped, black = frame_quality(frame)
        ph, th, td = visual_signatures(frame)
        metrics.append({"frame_idx": fi, "pts_time": float(frames[fi][0]),
                        "global": global_d, "local": local_d,
                        "text_global": text_global_d, "text_local": text_local_d,
                        "quality": q, "sharpness": sharp, "mean": mean, "clipped": clipped,
                        "black": black, "phash": ph, "text_hash": th, "text_density": td})
    return metrics

def select_burst(meta, event: Event, frames: dict[int,tuple[float,np.ndarray]], probe, segments):
    if not frames: return []
    metrics = _burst_metrics(frames)
    global_n = minmax(np.array([x["global"] for x in metrics]))
    local_n = minmax(np.array([x["local"] for x in metrics]))
    text_global_n = minmax(np.array([x["text_global"] for x in metrics]))
    text_local_n = minmax(np.array([x["text_local"] for x in metrics]))
    density_n = minmax(np.array([x["text_density"] for x in metrics]))
    quality = np.array([x["quality"] for x in metrics], np.float32)
    center = float(event.pts_time if event.pts_time is not None else np.mean([x["pts_time"] for x in metrics]))
    radius_s = max(1e-3, float(event.radius_ms or CONFIG["microburst"]["radius_ms"])/1000)
    proximity = np.asarray([math.exp(-abs(x["pts_time"]-center)/(0.5*radius_s)) for x in metrics], np.float32)
    if event.sources == {"common"}:
        evidence = 0.65*proximity + 0.35*quality
    elif "text_local_event" in event.sources:
        evidence = (0.05*global_n + 0.10*local_n + 0.10*text_global_n +
                    0.40*text_local_n + 0.15*density_n + 0.15*quality +
                    0.05*proximity)
    elif "text_event" in event.sources:
        evidence = (0.10*global_n + 0.15*local_n + 0.35*text_global_n +
                    0.15*text_local_n + 0.10*density_n + 0.10*quality +
                    0.05*proximity)
    else:
        evidence = (0.20*global_n + 0.35*local_n + 0.15*text_global_n +
                    0.15*density_n + 0.10*quality + 0.05*proximity)

    is_short = "short_shot" in event.sources
    duration_ms = max(0.0, float((event.end_time or center) - (event.start_time or center))*1000)
    order = list(np.argsort(evidence)[::-1])
    chosen = [order[0]]
    retain_max = CONFIG["microburst"]["retain_max"]
    if is_short and duration_ms <= CONFIG["microburst"]["very_short_segment_ms"]:
        retain_max = max(retain_max, 3)
        order = [0, int(np.argmax(quality)), len(metrics)-1] + order
        chosen = []
    for j in order:
        if j in chosen: continue
        if not chosen:
            chosen.append(j)
        else:
            if evidence[j] < CONFIG["microburst"]["second_min_score_ratio"]*evidence[chosen[0]]:
                continue
            distinct = all(
                hamming64(metrics[j]["phash"], metrics[k]["phash"]) > CONFIG["microburst"]["complement_phash_hamming"]
                or hamming64(metrics[j]["text_hash"], metrics[k]["text_hash"]) > CONFIG["microburst"]["complement_text_hamming"]
                for k in chosen
            )
            if distinct: chosen.append(j)
        if len(chosen) >= retain_max: break

    pframes = probe["frame_idx"].astype(np.int64)
    result = []
    for j in sorted(chosen, key=lambda x: metrics[x]["frame_idx"]):
        item = metrics[j]; fi = int(item["frame_idx"])
        pos = nearest_probe_pos(pframes, fi)
        sid = event.segment_id if event.segment_id >= 0 else segment_for_probe_pos(segments, pos)
        scores = dict(event.scores)
        scores.update({"burst_evidence": float(evidence[j]), "burst_global": item["global"],
                       "burst_local": item["local"],
                       "burst_text_global": item["text_global"],
                       "burst_text_local": item["text_local"]})
        result.append(FrameCandidate(
            frame_idx=fi, pts_time=float(item["pts_time"]), segment_id=sid,
            sources=set(event.sources)|{"microburst"}, priority=event.priority, scores=scores,
            quality=float(item["quality"]), phash=int(item["phash"]),
            text_hash=int(item["text_hash"]), text_density=float(item["text_density"]),
            embedding=np.asarray(probe["features"][pos], np.float32),
        ))
    return result

def refine_events(meta: VideoMeta, events: list[Event], probe: dict, segments):
    if not events: return []
    windows = []
    for event in events:
        center = float(event.pts_time if event.pts_time is not None else event.frame_idx/meta.fps)
        if "short_shot" in event.sources and event.start_time is not None and event.end_time is not None:
            start_t, end_t = max(0.0,event.start_time), event.end_time
        else:
            radius = float(event.radius_ms or CONFIG["microburst"]["radius_ms"])/1000
            start_t, end_t = max(0.0,center-radius), center+radius
        windows.append({"event":event, "start":start_t, "end":end_t, "frames":{}, "closed":False})
    windows.sort(key=lambda w:w["start"])
    container = av.open(meta.path); stream = container.streams.video[0]; stream.thread_type = "AUTO"
    target_w, target_h = scaled_hw(
        int(stream.codec_context.width or meta.width), int(stream.codec_context.height or meta.height),
        CONFIG["sentinel_width"],
    )
    refined = []
    active, next_window, decode_idx = [], 0, 0

    def finalize(window):
        if window["closed"]: return
        window["closed"] = True
        refined.extend(select_burst(meta, window["event"], window["frames"], probe, segments))
        window["frames"].clear()

    for av_frame in container.decode(video=0):
        pts_time = av_frame_time(av_frame, decode_idx, meta.fps)
        while next_window < len(windows) and windows[next_window]["start"] <= pts_time + 1e-9:
            active.append(windows[next_window]); next_window += 1
        still = []
        for window in active:
            if pts_time <= window["end"] + 1e-9: still.append(window)
            else: finalize(window)
        active = still
        if active:
            small = av_frame.reformat(width=target_w,height=target_h,format="bgr24").to_ndarray()
            for window in active:
                if window["start"]-1e-9 <= pts_time <= window["end"]+1e-9:
                    window["frames"][decode_idx] = (pts_time, small)
        decode_idx += 1
    container.close()
    for window in active: finalize(window)
    while next_window < len(windows): finalize(windows[next_window]); next_window += 1
    return refined


## 13. Candidate union + temporal/text-aware dedup

Không dedup global. Hai frame chỉ được coi là duplicate nếu cùng segment, gần ≤1 giây, cosine/pHash đều rất giống và text-state không đổi.


In [ ]:
def merge_same_frame(candidates: list[FrameCandidate]):
    merged = {}
    for c in candidates:
        if c.frame_idx not in merged:
            merged[c.frame_idx] = c; continue
        old = merged[c.frame_idx]
        old.sources |= c.sources
        old.priority = min(old.priority, c.priority)
        old.quality = max(old.quality, c.quality)
        old.btc_id = old.btc_id or c.btc_id
        for k,v in c.scores.items(): old.scores[k] = max(old.scores.get(k, -np.inf), v)
        if old.embedding is None: old.embedding = c.embedding
    return list(merged.values())

def is_text_state_same(a: FrameCandidate, b: FrameCandidate):
    cfg = CONFIG["dedup"]
    return (
        hamming64(a.text_hash, b.text_hash) <= cfg["text_hash_hamming"]
        and abs(a.text_density-b.text_density) <= cfg["text_density_delta"]
    )

def is_duplicate(a: FrameCandidate, b: FrameCandidate, fps: float):
    cfg = CONFIG["dedup"]
    if a.segment_id != b.segment_id: return False
    if abs(a.pts_time-b.pts_time)*1000 > cfg["max_temporal_distance_ms"]: return False
    if not is_text_state_same(a,b): return False
    if hamming64(a.phash,b.phash) > cfg["phash_hamming"]: return False
    if a.embedding is None or b.embedding is None: return False
    cosine = float(np.dot(a.embedding,b.embedding) /
                   ((np.linalg.norm(a.embedding)+1e-9)*(np.linalg.norm(b.embedding)+1e-9)))
    return cosine >= cfg["semantic_cosine"]

def source_rank(c: FrameCandidate):
    # BTC được ưu tiên trong cùng duplicate cluster; text/burst/short được bảo vệ qua P0 + text rule.
    btc_rank = 0 if ("btc" in c.sources and CONFIG["btc_union"]["prefer_btc"]) else 1
    return (c.priority, btc_rank, -c.quality, c.frame_idx)

def safe_dedup(candidates: list[FrameCandidate], fps: float):
    pool = merge_same_frame(candidates)
    selected = []
    for c in sorted(pool, key=source_rank):
        if not any(is_duplicate(c, kept, fps) for kept in selected):
            selected.append(c)
    return sorted(selected, key=lambda c: c.frame_idx)

def event_to_candidate(meta, event, probe, segments):
    pos = nearest_probe_pos(probe["frame_idx"], event.frame_idx)
    sid = event.segment_id if event.segment_id >= 0 else segment_for_probe_pos(segments, pos)
    c = candidate_from_probe(meta, probe, pos, sid, event.sources, event.priority, event.scores)
    c.frame_idx = event.frame_idx
    c.pts_time = float(event.pts_time if event.pts_time is not None else event.frame_idx/meta.fps)
    return c

def coverage_gap_fill(meta: VideoMeta, selected: list[FrameCandidate], probe: dict, segments):
    """Vá temporal gap sau dedup mà không thay thế adaptive/text/BTC frame."""
    cfg = CONFIG["coverage"]
    probe_times = np.asarray(probe["pts_time"], np.float64)
    probe_frames = np.asarray(probe["frame_idx"], np.int64)
    probe_quality = np.asarray(probe["quality"], np.float32)
    if not len(probe_times):
        return list(selected), {
            "coverage_added": 0, "max_temporal_gap_sec": None,
            "coverage_ok": False, "coverage_reason": "no_probe_frames",
        }

    timeline_start, timeline_end = float(probe_times[0]), float(probe_times[-1])
    max_gap = float(cfg["max_gap_sec"])
    step = float(cfg["target_step_sec"])
    tolerance = float(cfg["verify_tolerance_sec"])
    quality_weight = float(cfg["quality_weight"])
    if max_gap <= 0 or step <= 0 or step > max_gap:
        raise ValueError("coverage cần 0 < target_step_sec <= max_gap_sec")
    if not (0.0 <= quality_weight <= 1.0):
        raise ValueError("coverage quality_weight phải nằm trong [0,1]")

    adaptive = sorted(merge_same_frame(list(selected)), key=lambda c: (c.pts_time, c.frame_idx))
    used = {int(c.frame_idx) for c in adaptive}
    added = []

    def add_probe_after(left_time: float, right_limit: float):
        # Không chọn muộn hơn left+step, nên coverage không tạo gap trái > step.
        target = min(float(left_time) + step, float(right_limit))
        lo = int(np.searchsorted(probe_times, float(left_time) + 1e-9, side="left"))
        hi = int(np.searchsorted(probe_times, target + 1e-9, side="right"))
        eligible = [p for p in range(lo, hi) if int(probe_frames[p]) not in used]
        if not eligible:
            hard_cap = min(float(left_time) + max_gap, float(right_limit))
            hi = int(np.searchsorted(probe_times, hard_cap + 1e-9, side="right"))
            eligible = [p for p in range(lo, hi) if int(probe_frames[p]) not in used]
        if not eligible:
            return None
        eligible = np.asarray(eligible, np.int64)
        proximity = np.exp(
            -np.abs(probe_times[eligible] - target) / max(0.25*step, 1e-6)
        )
        score = (1.0-quality_weight)*proximity + quality_weight*probe_quality[eligible]
        pos = int(eligible[int(np.argmax(score))])
        sid = segment_for_probe_pos(segments, pos)
        c = candidate_from_probe(
            meta, probe, pos, sid, {"coverage_floor"}, 2,
            {
                "coverage_target_s": float(target),
                "coverage_probe_quality": float(probe_quality[pos]),
                "coverage_gap_left_s": float(left_time),
            },
        )
        used.add(int(c.frame_idx)); added.append(c)
        return c

    def fill_gap(left_time: float, right_time: float):
        cursor = float(left_time)
        guard = 0
        while float(right_time) - cursor > max_gap + 1e-9 and guard < len(probe_times):
            guard += 1
            c = add_probe_after(cursor, float(right_time) - 1e-9)
            if c is None:
                log(meta.video_id, "WARNING: không tìm được probe frame để vá coverage gap")
                break
            if float(c.pts_time) <= cursor + 1e-9:
                log(meta.video_id, "WARNING: coverage candidate không tiến thời gian")
                break
            cursor = float(c.pts_time)

    if bool(cfg["enabled"]):
        anchors = [timeline_start, *[float(c.pts_time) for c in adaptive], timeline_end]
        for left_time, right_time in zip(anchors[:-1], anchors[1:]):
            fill_gap(left_time, right_time)

    work = sorted(adaptive + added, key=lambda c: (c.pts_time, c.frame_idx))
    timeline_points = [timeline_start, *[float(c.pts_time) for c in work], timeline_end]
    max_actual = float(np.max(np.diff(timeline_points))) if len(timeline_points) >= 2 else 0.0
    ok = max_actual <= max_gap + tolerance
    if bool(cfg["enabled"]) and not ok:
        log(meta.video_id, f"WARNING: coverage max gap={max_actual:.3f}s > {max_gap+tolerance:.3f}s")
    return work, {
        "coverage_added": len(added),
        "max_temporal_gap_sec": max_actual,
        "coverage_ok": bool(ok),
        "coverage_reason": "enabled" if bool(cfg["enabled"]) else "disabled",
    }


## 13b. L25 static-state consolidation

Chỉ L25 chạy stage này, sau coverage và trước materialize. Candidate được decode lại ở
low resolution để đo localized grayscale/edge change theo grid 12×20. Một static run
chỉ hợp lệ khi hash, text density, SigLIP, local pixels, local edges và motion đồng thời
ổn định. Stage luôn giữ frame đầu/cuối, frame nét nhất, protected trigger và heartbeat;
nếu conditional coverage verification lỗi thì fallback nguyên bộ frame trước compression.


In [ ]:
def _candidate_cosine(a: FrameCandidate, b: FrameCandidate) -> float:
    if a.embedding is None or b.embedding is None:
        return -1.0
    av = np.asarray(a.embedding, np.float32)
    bv = np.asarray(b.embedding, np.float32)
    return float(np.dot(av, bv)/((np.linalg.norm(av)+1e-9)*(np.linalg.norm(bv)+1e-9)))

def _nearest_array_values(source_idx, source_values, targets):
    source_idx = np.asarray(source_idx, np.int64)
    source_values = np.asarray(source_values, np.float32)
    targets = np.asarray(targets, np.int64)
    pos = np.searchsorted(source_idx, targets)
    pos = np.clip(pos, 0, max(0, len(source_idx)-1))
    prev = np.clip(pos-1, 0, max(0, len(source_idx)-1))
    choose_prev = np.abs(source_idx[prev]-targets) <= np.abs(source_idx[pos]-targets)
    pos = np.where(choose_prev, prev, pos)
    return source_values[pos]

def scan_l25_candidate_states(meta, candidates, sentinel):
    cfg = CONFIG["l25_static_consolidation"]
    work = sorted(candidates, key=lambda c: c.frame_idx)
    target_positions = {int(c.frame_idx):i for i,c in enumerate(work)}
    target_frames = np.asarray([c.frame_idx for c in work], np.int64)
    motion = _nearest_array_values(
        sentinel["frame_idx"], sentinel["motion_raw"], target_frames
    )
    metrics = [None]*len(work)
    rows, cols = cfg["grid"]
    top_k = int(cfg["top_tiles"])
    prev_gray = prev_edges = None
    prev_candidate = None
    guard_failures = defaultdict(int)

    container = av.open(meta.path)
    try:
        stream = container.streams.video[0]; stream.thread_type = "AUTO"
        target_w, target_h = scaled_hw(
            int(stream.codec_context.width or meta.width),
            int(stream.codec_context.height or meta.height),
            int(cfg["analysis_width"]),
        )
        decode_idx = 0; last_target = int(target_frames[-1]) if len(target_frames) else -1
        for av_frame in container.decode(video=0):
            if decode_idx > last_target: break
            position = target_positions.get(decode_idx)
            if position is not None:
                image = av_frame.reformat(
                    width=target_w, height=target_h, format="bgr24"
                ).to_ndarray()
                gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
                edges = full_text_edge_map(gray)
                quality, sharpness, _, _, _ = frame_quality(image)
                phash = _hash64_from_gray(gray)
                text_hash = _hash64_from_gray(edges)
                text_density = float((edges > 0).mean())
                candidate = work[position]
                item = {
                    "quality":float(quality), "sharpness":float(sharpness),
                    "phash":int(phash), "text_hash":int(text_hash),
                    "text_density":text_density, "motion":float(motion[position]),
                    "same_previous":False,
                }
                if prev_gray is not None:
                    gray_diff = cv2.absdiff(gray, prev_gray).astype(np.float32)/255.0
                    edge_diff = cv2.absdiff(edges, prev_edges).astype(np.float32)/255.0
                    values = {
                        "input_gap":float(candidate.pts_time-prev_candidate.pts_time),
                        "phash_hamming":hamming64(phash, metrics[position-1]["phash"]),
                        "text_hash_hamming":hamming64(text_hash, metrics[position-1]["text_hash"]),
                        "text_density_delta":abs(text_density-metrics[position-1]["text_density"]),
                        "semantic_cosine":_candidate_cosine(prev_candidate, candidate),
                        "global_gray_delta":float(gray_diff.mean()),
                        "local_gray_delta":tile_top_delta(gray_diff, rows, cols, top_k),
                        "global_edge_delta":float(edge_diff.mean()),
                        "local_edge_delta":tile_top_delta(edge_diff, rows, cols, top_k),
                        "motion_pair":max(float(motion[position-1]),float(motion[position])),
                    }
                    guards = {
                        "input_gap": values["input_gap"] <= cfg["max_input_gap_sec"],
                        "phash": values["phash_hamming"] <= cfg["phash_hamming"],
                        "text_hash": values["text_hash_hamming"] <= cfg["text_hash_hamming"],
                        "text_density": values["text_density_delta"] <= cfg["text_density_delta"],
                        "semantic": values["semantic_cosine"] >= cfg["semantic_cosine"],
                        "global_gray": values["global_gray_delta"] <= cfg["global_gray_delta"],
                        "local_gray": values["local_gray_delta"] <= cfg["local_gray_delta"],
                        "global_edge": values["global_edge_delta"] <= cfg["global_edge_delta"],
                        "local_edge": values["local_edge_delta"] <= cfg["local_edge_delta"],
                        "motion": values["motion_pair"] <= cfg["motion_max_raw"],
                    }
                    for name, passed in guards.items():
                        if not passed: guard_failures[name] += 1
                    item.update(values)
                    item["same_previous"] = bool(all(guards.values()))
                metrics[position] = item
                prev_gray, prev_edges, prev_candidate = gray, edges, candidate
            decode_idx += 1
    finally:
        container.close()
    if any(x is None for x in metrics):
        missing = [work[i].frame_idx for i,x in enumerate(metrics) if x is None]
        raise RuntimeError(f"L25 state scan thiếu {len(missing)} target frame: {missing[:10]}")
    return work, metrics, dict(guard_failures)

def _l25_is_protected(candidate):
    protected = set(CONFIG["l25_static_consolidation"]["protected_sources"])
    return candidate.priority == 0 or bool(candidate.sources & protected)

def _l25_choose_run_anchors(work, metrics, start, end, cfg):
    run_positions = list(range(start,end+1))
    selected = {start,end}
    roles = defaultdict(set)
    roles[start].add("first"); roles[end].add("last")
    best = max(run_positions, key=lambda i: (metrics[i]["quality"],-work[i].frame_idx))
    selected.add(best); roles[best].add("best")
    for i in run_positions:
        if _l25_is_protected(work[i]):
            selected.add(i); roles[i].add("protected")

    # Fill mọi gap giữa mandatory anchors bằng representative gần heartbeat nhất,
    # ưu tiên quality nhưng không bao giờ vượt max_static_gap_sec.
    max_gap = float(cfg["max_static_gap_sec"])
    heartbeat = float(cfg["heartbeat_sec"])
    while True:
        ordered = sorted(selected)
        gap_pair = next((
            (left,right) for left,right in zip(ordered[:-1],ordered[1:])
            if work[right].pts_time-work[left].pts_time > max_gap+1e-9
        ), None)
        if gap_pair is None: break
        left,right = gap_pair
        target_time = min(work[left].pts_time+heartbeat, work[right].pts_time)
        eligible = [
            i for i in range(left+1,right)
            if work[i].pts_time <= work[left].pts_time+max_gap
        ]
        if not eligible:
            return None, None
        chosen = max(eligible, key=lambda i: (
            -abs(work[i].pts_time-target_time)+0.15*heartbeat*metrics[i]["quality"],
            metrics[i]["quality"], -work[i].frame_idx,
        ))
        selected.add(chosen); roles[chosen].add("heartbeat")
    return selected, roles

def l25_static_consolidation(meta, candidates, sentinel):
    cfg = CONFIG["l25_static_consolidation"]
    original = sorted(merge_same_frame(list(candidates)), key=lambda c:c.frame_idx)
    base_stats = {
        "l25_static_enabled":False, "l25_static_input_frames":len(original),
        "l25_static_output_frames":len(original), "l25_static_removed":0,
        "l25_static_runs":0, "l25_static_reduction":0.0,
        "l25_static_conditional_coverage_ok":True,
    }
    if not cfg.get("enabled",True) or meta.category not in set(cfg.get("categories",["L25"])):
        return original, base_stats
    if len(original) < int(cfg["min_run_frames"]):
        base_stats["l25_static_enabled"] = True
        return original, base_stats

    work, metrics, guard_failures = scan_l25_candidate_states(meta, original, sentinel)
    raw_runs, start = [], 0
    for i in range(1,len(work)):
        if not metrics[i]["same_previous"]:
            raw_runs.append((start,i-1)); start = i
    raw_runs.append((start,len(work)-1))

    kept = set(range(len(work)))
    run_by_position, roles_by_position, run_details = {}, defaultdict(set), {}
    run_id = 0
    for a,b in raw_runs:
        duration = float(work[b].pts_time-work[a].pts_time)
        if b-a+1 < int(cfg["min_run_frames"]) or duration < float(cfg["static_min_sec"]):
            continue
        chosen, roles = _l25_choose_run_anchors(work,metrics,a,b,cfg)
        if chosen is None: continue
        removed = set(range(a,b+1))-set(chosen)
        if not removed: continue
        kept -= removed
        for pos in chosen:
            run_by_position[pos] = run_id
            roles_by_position[pos] |= roles[pos]
        run_details[run_id] = {
            "duration":duration, "input":b-a+1,
            "output":len(chosen), "removed":len(removed),
        }
        run_id += 1

    selected_positions = sorted(kept)
    protected_positions = {i for i,c in enumerate(work) if _l25_is_protected(c)}
    protected_ok = protected_positions.issubset(kept)
    normal_limit = (
        float(CONFIG["coverage"]["max_gap_sec"])
        + float(CONFIG["coverage"]["verify_tolerance_sec"])
    )
    static_limit = (
        float(cfg["max_static_gap_sec"])
        + float(CONFIG["coverage"]["verify_tolerance_sec"])
    )
    violations, gaps, dynamic_gaps = [], [], []
    for left,right in zip(selected_positions[:-1],selected_positions[1:]):
        gap = float(work[right].pts_time-work[left].pts_time); gaps.append(gap)
        same_run = (
            left in run_by_position and right in run_by_position
            and run_by_position[left] == run_by_position[right]
        )
        if same_run:
            if gap > static_limit: violations.append((left,right,gap,"static"))
        else:
            dynamic_gaps.append(gap)
            if gap > normal_limit: violations.append((left,right,gap,"dynamic"))

    verify_ok = bool(protected_ok and not violations)
    if not verify_ok and cfg.get("fallback_on_verify_failure",True):
        log(meta.video_id, f"WARNING: L25 consolidation verify failed; fallback "
            f"protected_ok={protected_ok}, violations={violations[:5]}")
        return original, {
            **base_stats, "l25_static_enabled":True,
            # Output đã fallback nguyên bộ pre-consolidation nên coverage gốc vẫn an toàn.
            "l25_static_conditional_coverage_ok":True,
            "l25_static_verify_failure":True,
            "l25_static_fallback":True,
            "l25_static_guard_failures":guard_failures,
        }

    for pos in selected_positions:
        rid = run_by_position.get(pos)
        if rid is None: continue
        candidate = work[pos]
        detail = run_details[rid]
        candidate.sources.add("l25_static_anchor")
        candidate.scores.update({
            "l25_static_verified":1.0, "l25_static_run_id":float(rid),
            "l25_static_run_duration_s":float(detail["duration"]),
            "l25_static_removed_in_run":float(detail["removed"]),
        })
        for role in roles_by_position[pos]:
            candidate.sources.add(f"l25_static_{role}")

    consolidated = [work[i] for i in selected_positions]
    removed_count = len(work)-len(consolidated)
    stats = {
        "l25_static_enabled":True,
        "l25_static_input_frames":len(work),
        "l25_static_output_frames":len(consolidated),
        "l25_static_removed":removed_count,
        "l25_static_runs":len(run_details),
        "l25_static_reduction":removed_count/max(len(work),1),
        "l25_static_protected_frames":len(protected_positions),
        "l25_static_max_gap_sec":max(gaps) if gaps else 0.0,
        "l25_static_max_dynamic_gap_sec":max(dynamic_gaps) if dynamic_gaps else 0.0,
        "l25_static_conditional_coverage_ok":verify_ok,
        "l25_static_guard_failures":guard_failures,
        "l25_static_fallback":False,
    }
    log(meta.video_id, f"L25 static consolidation: {len(work):,}→{len(consolidated):,} "
        f"(-{removed_count:,}, {stats['l25_static_reduction']:.1%}), "
        f"runs={len(run_details)}, max_gap={stats['l25_static_max_gap_sec']:.2f}s")
    return consolidated, stats


## 14. Materialize JPEG, frame registry và embeddings tuỳ chọn

Canonical image path: `keyframes/L21/L21_V001/f00018372.jpg`. Full metadata nằm trong `registry`; `map-keyframes/*.csv` giữ schema legacy **đúng bốn cột theo đúng thứ tự** `n,pts_time,fps,frame_idx` để tương thích `keyframe_mapping.py`.

Mặc định chỉ decode/lưu JPEG và registry. `save_selector_embeddings=False` tránh một
lượt SigLIP thứ hai trên hàng nghìn retained frame; việc chọn keyframe và recall không
đổi. Nếu cần exact SigLIP embedding để benchmark/debug, bật lại cờ này.


In [ ]:
def nearest_btc(c: FrameCandidate, btc: list[FrameCandidate]):
    if c.btc_id: return c.btc_id
    if not btc: return None
    return min(btc, key=lambda x: abs(x.frame_idx-c.frame_idx)).btc_id

def materialize(meta: VideoMeta, final: list[FrameCandidate], selector, final_encoder,
                output_video_id: Optional[str] = None, save_images=True,
                save_selector_embeddings=True, save_final_embeddings=True):
    out_vid = output_video_id or meta.video_id
    category = meta.category
    image_dir = OUT / "keyframes" / category / out_vid
    if save_images:
        if image_dir.exists(): shutil.rmtree(image_dir)
        image_dir.mkdir(parents=True, exist_ok=True)
    targets = {c.frame_idx: c for c in final}
    need_final = bool(save_final_embeddings and final_encoder is not None)
    need_selector = bool(save_selector_embeddings or (need_final and final_encoder is selector))
    need_decode = bool(save_images or need_selector or need_final)
    saved = []
    selector_by_frame, final_by_frame = {}, {}
    selector_images, selector_candidates = [], []
    final_images, final_candidates = [], []

    def flush_selector():
        nonlocal selector_images, selector_candidates
        if not selector_images: return
        vectors = selector.encode(selector_images)
        for c, vector in zip(selector_candidates, vectors): selector_by_frame[c.frame_idx] = vector
        selector_images, selector_candidates = [], []

    def flush_pe():
        nonlocal final_images, final_candidates
        if not final_images: return
        vectors = final_encoder.encode(final_images)
        for c, vector in zip(final_candidates, vectors): final_by_frame[c.frame_idx] = vector
        final_images, final_candidates = [], []

    if need_decode:
        container = av.open(meta.path); stream = container.streams.video[0]; stream.thread_type = "AUTO"
        decode_idx = 0; last = max(targets) if targets else -1
        for av_frame in container.decode(video=0):
            if decode_idx > last: break
            if decode_idx in targets:
                rgb = av_frame.to_ndarray(format="rgb24")
                bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
                bgr = resize_max_side(bgr, CONFIG["save_max_side"])
                c = targets[decode_idx]
                c.pts_time = av_frame_time(av_frame, decode_idx, meta.fps)
                q, _, _, _, _ = frame_quality(bgr); ph, th, td = visual_signatures(bgr)
                c.quality, c.phash, c.text_hash, c.text_density = q, ph, th, td
                if save_images:
                    image_path = image_dir / f"f{decode_idx:08d}.jpg"
                    cv2.imwrite(str(image_path), bgr, [cv2.IMWRITE_JPEG_QUALITY, CONFIG["jpg_quality"]])
                    c.scores["_image_path"] = str(image_path)
                if need_selector:
                    selector_images.append(bgr); selector_candidates.append(c)
                    if len(selector_images) >= selector.total_batch_size: flush_selector()
                if need_final:
                    if final_encoder is selector:
                        pass
                    else:
                        final_images.append(bgr); final_candidates.append(c)
                        if len(final_images) >= final_encoder.total_batch_size: flush_pe()
                saved.append(c)
            decode_idx += 1
        container.close(); flush_selector()
        if need_final and final_encoder is not selector: flush_pe()
        if need_final and final_encoder is selector: final_by_frame = dict(selector_by_frame)
    else:
        saved = list(final)
    if not saved: raise RuntimeError(f"Không materialize được frame nào cho {meta.video_id}")

    btc = [c for c in saved if "btc" in c.sources]
    rows = []
    for n, c in enumerate(saved, 1):
        frame_id = f"{meta.video_id}@f{c.frame_idx:08d}"
        rows.append({
            "n": n, "frame_id": frame_id, "video_id": meta.video_id,
            "category": category, "frame_idx": int(c.frame_idx),
            "pts_time": round(c.pts_time, 6), "timestamp": sec_to_hms(c.pts_time),
            "fps": round(meta.fps, 8), "segment_id": int(c.segment_id),
            "sources": "|".join(sorted(c.sources)), "priority": f"P{c.priority}",
            "is_btc": "btc" in c.sources, "btc_id": c.btc_id,
            "nearest_btc_id": nearest_btc(c, btc),
            "quality": round(float(c.quality), 6),
            "phash_hex": f"{int(c.phash):016x}", "text_hash_hex": f"{int(c.text_hash):016x}",
            "text_density": round(float(c.text_density), 6),
            "l25_static_verified": bool(c.scores.get("l25_static_verified",0.0) > 0),
            "l25_static_run_id": (
                int(c.scores["l25_static_run_id"])
                if "l25_static_run_id" in c.scores else -1
            ),
            "scores_json": stable_json({k:round(float(v),6) for k,v in c.scores.items() if not k.startswith("_")}),
            "image_path": c.scores.get("_image_path"),
            "source_video_path": meta.path,
            "normalized_video_dir": f"Videos_{category}",
            "legacy_keyframe_id": f"{meta.video_id}#{n:03d}",
        })
    df = pd.DataFrame(rows)
    registry_path = OUT / "registry" / f"{out_vid}.csv"
    df.to_csv(registry_path, index=False)
    df.to_json(OUT / "registry" / f"{out_vid}.jsonl", orient="records", lines=True, force_ascii=False)
    legacy_columns = ["n", "pts_time", "fps", "frame_idx"]
    df.loc[:, legacy_columns].to_csv(
        OUT / "map-keyframes" / f"{out_vid}.csv", index=False
    )
    if need_selector and selector_by_frame:
        np.save(OUT / "selector-embeddings" / f"{out_vid}.npy",
                np.stack([selector_by_frame[c.frame_idx] for c in saved]).astype(np.float16))
    if need_final and final_by_frame:
        np.save(OUT / "final-embeddings" / f"{out_vid}.npy",
                np.stack([final_by_frame[c.frame_idx] for c in saved]).astype(np.float16))
    return df


## 15. Pipeline một video + resume


In [ ]:
def success_path(video_id): return OUT / "registry" / f"{video_id}._SUCCESS.json"

def already_done(video_id):
    path = success_path(video_id)
    if not path.exists() or not CONFIG["skip_existing"]: return False
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        registry = OUT / "registry" / f"{video_id}.csv"
        image_dir = OUT / "keyframes" / video_id.split("_",1)[0] / video_id
        archive = OUT / "archives" / f"{video_id}.keyframes.zip"
        archive_meta = OUT / "archives" / f"{video_id}.archive.json"
        assets_expected = not CONFIG["storage"]["batch_registry_only"]
        loose_ok = image_dir.exists() and any(image_dir.glob("*.jpg"))
        archive_ok = archive.exists() and archive_meta.exists()
        assets_ok = (loose_ok or archive_ok) if assets_expected else True
        return payload.get("pipeline_signature") == PIPELINE_SIGNATURE and registry.exists() and assets_ok
    except Exception:
        return False

def process_video(path: str, video_id: str, selector, final_encoder,
                  force=False, max_seconds=None, output_tag=""):
    out_vid = video_id + output_tag
    if not force and not output_tag and already_done(video_id):
        return {"video_id": video_id, "status": "skip"}
    started = time.time()
    meta = probe_video(path, video_id)
    log(video_id, f"codec={meta.codec}, fps={meta.fps:.4f}, duration={meta.duration_sec/60:.2f}m")
    # force chỉ rebuild selection/output; pass-1 cache đúng fingerprint vẫn được reuse.
    probe, sentinel = fused_pass1(meta, selector, max_seconds=max_seconds, reuse_cache=True)
    effective_total = int(sentinel["frame_idx"][-1]) + 1
    effective_duration = float(sentinel["pts_time"][-1]) if len(sentinel["pts_time"]) else 0.0
    meta = VideoMeta(**{**meta.__dict__, "total_frames": effective_total,
                        "duration_sec": effective_duration})

    sent_events, sentinel_z = sentinel_events(meta, sentinel)
    segments, boundary_score, boundary_z, best_window, boundary_events = build_segments(
        meta, probe, sentinel, sent_events
    )
    info_candidates, semantic_events, short_events = infoshoot_candidates(meta, probe, segments)

    # Gán segment cho native events và giữ trigger gốc trong safety pool.
    for event in sent_events:
        pos = nearest_probe_pos(probe["frame_idx"], event.frame_idx)
        event.segment_id = segment_for_probe_pos(segments, pos)
    event_radius = max(1, int(round(CONFIG["infoshoot"]["temporal_nms_ms"]*meta.fps/1000)))
    all_events = merge_events(sent_events + semantic_events + short_events + boundary_events, event_radius)
    trigger_candidates = [event_to_candidate(meta,e,probe,segments) for e in all_events]
    refined = refine_events(meta, all_events, probe, segments)
    btc = [] if max_seconds else load_btc_candidates(meta, probe, segments)
    pool = info_candidates + trigger_candidates + refined + btc
    adaptive_final = safe_dedup(pool, meta.fps)
    # Coverage chạy sau dedup: chỉ vá gap, không cạnh tranh với adaptive/BTC frame.
    final, coverage_stats = coverage_gap_fill(meta, adaptive_final, probe, segments)
    pre_static_frames = len(final)
    final, l25_static_stats = l25_static_consolidation(meta, final, sentinel)
    if l25_static_stats.get("l25_static_enabled"):
        coverage_stats["pre_consolidation_max_temporal_gap_sec"] = coverage_stats.get(
            "max_temporal_gap_sec"
        )
        coverage_stats["max_temporal_gap_sec"] = l25_static_stats.get(
            "l25_static_max_gap_sec", coverage_stats.get("max_temporal_gap_sec")
        )
        coverage_stats["coverage_ok"] = bool(
            l25_static_stats.get("l25_static_conditional_coverage_ok",False)
        )
        coverage_stats["coverage_reason"] = (
            "l25_fallback_original_coverage"
            if l25_static_stats.get("l25_static_fallback",False)
            else "l25_dynamic_0.5s_static_heartbeat"
        )
    registry_only = CONFIG["run_mode"] == "batch" and CONFIG["storage"]["batch_registry_only"]
    df = materialize(
        meta, final, selector, final_encoder, out_vid,
        save_images=not registry_only,
        save_selector_embeddings=CONFIG["storage"]["save_selector_embeddings"] and not registry_only,
        save_final_embeddings=CONFIG["storage"]["save_final_embeddings"] and not registry_only,
    )

    source_counts = defaultdict(int)
    for sources in df.sources:
        for source in str(sources).split("|"): source_counts[source] += 1
    result = {
        "video_id": video_id, "output_video_id": out_vid, "status": "ok",
        "codec": meta.codec, "fps": meta.fps, "duration_sec": meta.duration_sec,
        "native_frames": len(sentinel["frame_idx"]), "probe_frames": len(probe["frame_idx"]),
        "segments": len(segments), "sentinel_events": len(sent_events),
        "semantic_events": len(semantic_events), "boundary_events": len(boundary_events),
        "short_segments": len(short_events),
        "btc_frames": len(btc), "pool_candidates": len(merge_same_frame(pool)),
        "adaptive_frames": len(adaptive_final),
        "pre_static_frames": pre_static_frames,
        **coverage_stats,
        **l25_static_stats,
        "final_frames": len(df), "retained_fps": len(df)/max(meta.duration_sec,1e-9),
        "selector": selector.kind, "final_encoder": final_encoder.kind if final_encoder else None,
        "registry_only": registry_only,
        "source_counts": dict(source_counts), "elapsed_sec": time.time()-started,
        "pipeline_signature": PIPELINE_SIGNATURE,
    }
    atomic_json(OUT / "registry" / f"{out_vid}.stats.json", result)
    if not output_tag:
        atomic_json(success_path(video_id), result)
    log(video_id, f"DONE final={len(df)}, retained={result['retained_fps']:.3f} fps, "
        f"time={result['elapsed_sec']/60:.1f}m")
    return result

def selected_video_rows():
    rows = VIDEOS.copy().reset_index(drop=True)
    rows["_scope_index"] = np.arange(len(rows), dtype=np.int64)
    a = int(CONFIG["range_start"])
    b = CONFIG["range_end"]
    rows = rows.iloc[a:b].copy()
    n, shard = int(CONFIG["num_shards"]), int(CONFIG["shard_index"])
    if not (0 <= shard < n): raise ValueError("shard_index phải nằm trong [0, num_shards)")
    rows = rows.iloc[shard::n].reset_index(drop=True)
    return rows


## 16. Khởi tạo model

Nếu chỉ muốn audit đường dẫn, đặt `run_mode="audit"`; cell sẽ không tải model.


In [ ]:
SELECTOR = FINAL_ENCODER = None
if CONFIG["run_mode"] not in {"audit", "merge"}:
    if CONFIG["run_mode"] == "batch":
        assert_full_dataset_ready_for_batch()
    SELECTOR = build_selector()
    log("Selector:", SELECTOR.kind, "dim=", SELECTOR.dim,
        "batch_total=", SELECTOR.total_batch_size)
    log("Throughput:",
        f"native={CONFIG['performance']['native_backend']}",
        f"native_batch={CONFIG['performance']['native_batch_frames']}",
        f"GPU SigLIP preprocess={CONFIG['performance']['gpu_siglip_preprocess']}",
        f"motion={CONFIG['motion_fps']} fps",
        f"save selector embeddings={CONFIG['storage']['save_selector_embeddings']}")
    FINAL_ENCODER = build_final_encoder(SELECTOR)
    log("Final encoder:", FINAL_ENCODER.kind if FINAL_ENCODER else "disabled")


## 17. Smoke test — chạy FULL nhiều video do bạn chỉ định

Notebook validate toàn bộ `CONFIG["smoke_video_paths"]` trước, sau đó chạy tuần tự từng
video từ frame đầu đến cuối. Mỗi output có hậu tố `__smoke`, không đánh dấu video thật là
completed nên batch vẫn xử lý lại bình thường. Một video lỗi được ghi vào `errors/` và không
ngăn các smoke video còn lại chạy tiếp.


In [ ]:
def resolve_smoke_items(path_values):
    if isinstance(path_values, (str, Path)):
        raise TypeError(
            "CONFIG['smoke_video_paths'] phải là list đường dẫn, ví dụ [\"...mp4\"]."
        )
    if not isinstance(path_values, (list, tuple)) or not path_values:
        raise ValueError("CONFIG['smoke_video_paths'] phải là list không rỗng.")

    items, seen = [], {}
    for raw_path in path_values:
        smoke_path = Path(raw_path)
        if not smoke_path.is_file():
            raise FileNotFoundError(
                f"Không thấy smoke video: {smoke_path}\n"
                "Hãy copy đúng path từ Kaggle Dataset panel."
            )
        reason = excluded_reason(smoke_path)
        if reason:
            raise ValueError(f"Smoke path nằm trong folder đã exclude: {reason}")
        video_id = canonical_video_id(smoke_path.name)
        if video_id.split("_", 1)[0] not in CONFIG["categories"]:
            raise ValueError(f"Smoke video không thuộc K01..K20 hoặc L21..L30: {video_id}")
        if video_id in seen:
            raise ValueError(
                f"Hai smoke path cùng canonical video_id={video_id}:\n"
                f"- {seen[video_id]}\n- {smoke_path}"
            )
        seen[video_id] = smoke_path
        items.append({"video_id": video_id, "path": smoke_path})
    return items

SMOKE_RUNS = []
SMOKE_RESULTS = []
SMOKE_RESULTS_DF = pd.DataFrame()
if CONFIG["run_mode"] == "smoke":
    smoke_items = resolve_smoke_items(CONFIG["smoke_video_paths"])
    print(f"Smoke queue: {len(smoke_items)} full video(s)")
    for smoke_i, item in enumerate(smoke_items, 1):
        smoke_path, smoke_video_id = item["path"], item["video_id"]
        print("\n" + "="*96)
        print(f"SMOKE {smoke_i}/{len(smoke_items)} · {smoke_video_id}")
        print("="*96)
        log("FULL smoke video:", smoke_path)
        try:
            result = process_video(
                str(smoke_path), smoke_video_id, SELECTOR, FINAL_ENCODER,
                force=True, max_seconds=None, output_tag="__smoke",
            )
            registry_path = OUT / "registry" / f"{smoke_video_id}__smoke.csv"
            smoke_df = pd.read_csv(registry_path)
            SMOKE_RUNS.append({
                "video_id": smoke_video_id, "path": str(smoke_path),
                "result": result, "registry_path": registry_path,
            })
            SMOKE_RESULTS.append(result)
            print(f"Đã trích {len(smoke_df):,} keyframe. Registry: {registry_path}")
            display(pd.DataFrame([result]))
            display(smoke_df[[
                "frame_idx", "timestamp", "sources", "priority", "quality", "image_path"
            ]])
        except Exception as exc:
            traceback.print_exc()
            failed = {
                "video_id": smoke_video_id, "status": f"error:{type(exc).__name__}",
                "error": str(exc), "path": str(smoke_path),
            }
            SMOKE_RESULTS.append(failed)
            atomic_json(OUT/"errors"/f"{smoke_video_id}__smoke.error.json", {
                **failed, "traceback": traceback.format_exc(),
            })
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    SMOKE_RESULTS_DF = pd.DataFrame(SMOKE_RESULTS)
    print("\n=== MULTI-SMOKE SUMMARY ===")
    display(SMOKE_RESULTS_DF)
    if not SMOKE_RUNS:
        raise RuntimeError("Tất cả smoke videos đều lỗi; xem thư mục errors/.")


## 18. Smoke diagnostics + gallery ngay trong cell output

Cell hiển thị source distribution, temporal-gap statistics, các khoảng trống lớn nhất, một
contact sheet sample đều toàn timeline và slider để duyệt **toàn bộ** keyframe. Gallery slider
chỉ load một trang mỗi lần nên không làm Kaggle UI treo.


In [ ]:
import matplotlib.pyplot as plt

def smoke_diagnostics(registry_csv):
    df = pd.read_csv(registry_csv).sort_values("pts_time").reset_index(drop=True)
    print(f"Retained: {len(df):,} keyframe | timeline={sec_to_hms(df.pts_time.max() if len(df) else 0)}")
    source_counts = df["sources"].fillna("").str.split("|").explode().value_counts()
    display(source_counts.rename("count").to_frame())
    if len(df) >= 2:
        gaps = np.diff(df.pts_time.to_numpy(float))
        display(pd.Series(gaps).describe(percentiles=[.5,.75,.9,.95,.99]).rename("gap_seconds").to_frame())
        allowed = CONFIG["coverage"]["max_gap_sec"] + CONFIG["coverage"]["verify_tolerance_sec"]
        if CONFIG["coverage"]["enabled"]:
            is_l25_static = (
                len(df) and str(df.category.iloc[0]) == "L25"
                and "l25_static_run_id" in df and "l25_static_verified" in df
            )
            bad = []
            if is_l25_static:
                static_allowed = (
                    CONFIG["l25_static_consolidation"]["max_static_gap_sec"]
                    + CONFIG["coverage"]["verify_tolerance_sec"]
                )
                for i,gap in enumerate(gaps):
                    same_static_run = (
                        bool(df.iloc[i].l25_static_verified)
                        and bool(df.iloc[i+1].l25_static_verified)
                        and int(df.iloc[i].l25_static_run_id) >= 0
                        and int(df.iloc[i].l25_static_run_id) == int(df.iloc[i+1].l25_static_run_id)
                    )
                    if gap > (static_allowed if same_static_run else allowed):
                        bad.append((i,float(gap)))
                mark = "✓" if not bad else "⚠"
                print(f"{mark} Conditional coverage: dynamic≤{allowed:.3f}s, "
                      f"verified-static≤{static_allowed:.3f}s; violations={len(bad)}")
            else:
                mark = "✓" if float(gaps.max()) <= allowed else "⚠"
                print(f"{mark} Coverage max retained gap={float(gaps.max()):.3f}s; bound≈{allowed:.3f}s")
        largest = np.argsort(-gaps)[:min(15,len(gaps))]
        display(pd.DataFrame([{
            "gap_s":round(float(gaps[i]),3),
            "from":sec_to_hms(df.iloc[i].pts_time), "to":sec_to_hms(df.iloc[i+1].pts_time),
            "from_sources":df.iloc[i].sources, "to_sources":df.iloc[i+1].sources,
        } for i in largest]))
    return df

def preview_uniform_timeline(registry_csv, max_items=120, cols=6):
    full = pd.read_csv(registry_csv).sort_values("pts_time").reset_index(drop=True)
    if max_items is not None and len(full) > max_items:
        positions = np.unique(np.linspace(0,len(full)-1,max_items).round().astype(int))
        sample = full.iloc[positions].reset_index(drop=True)
        print(f"Contact sheet sample đều {len(sample)}/{len(full)} keyframe.")
    else:
        sample = full
    rows = max(1,math.ceil(len(sample)/cols))
    fig, axes = plt.subplots(rows,cols,figsize=(3.2*cols,2.8*rows))
    axes = np.atleast_1d(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for ax,(_,row) in zip(axes,sample.iterrows()):
        image = cv2.imread(str(row.image_path))
        if image is not None: ax.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB))
        ax.set_title(f"f{int(row.frame_idx)} · {row.timestamp}\n{row.sources}",fontsize=8)
    plt.tight_layout(); plt.show()

def preview_registry_page(registry_csv, page=1, page_size=36, cols=6):
    full = pd.read_csv(registry_csv)
    total_pages = max(1, math.ceil(len(full) / page_size))
    page = int(np.clip(page, 1, total_pages))
    start = (page - 1) * page_size
    df = full.iloc[start:start + page_size]
    rows = max(1, math.ceil(len(df)/cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.2*cols, 2.8*rows))
    axes = np.atleast_1d(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for ax, (_, row) in zip(axes, df.iterrows()):
        image = cv2.imread(str(row.image_path))
        if image is not None: ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(
            f"#{int(row.n):03d} · f{int(row.frame_idx)} · {row.timestamp}\n"
            f"{row.sources} · q={float(row.quality):.2f}", fontsize=8
        )
        ax.axis("off")
    fig.suptitle(f"Trang {page}/{total_pages} · keyframe {start+1}–{start+len(df)} / {len(full)}", y=1.002)
    plt.tight_layout(); plt.show()

def interactive_keyframe_gallery(registry_csv, page_size=36, cols=6):
    total = len(pd.read_csv(registry_csv, usecols=["frame_idx"]))
    total_pages = max(1, math.ceil(total/page_size))
    print(f"Gallery: {total:,} keyframe · {total_pages} trang · {page_size} ảnh/trang")
    try:
        from ipywidgets import IntSlider, interact
        interact(
            lambda page: preview_registry_page(registry_csv, page, page_size, cols),
            page=IntSlider(value=1, min=1, max=total_pages, step=1,
                           description="Trang", continuous_update=False),
        )
    except Exception as exc:
        print(f"ipywidgets không khả dụng ({exc}); hiển thị trang đầu.")
        print(f"Xem trang khác bằng: preview_registry_page({str(registry_csv)!r}, page=2)")
        preview_registry_page(registry_csv, 1, page_size, cols)

if CONFIG["run_mode"] == "smoke" and SMOKE_RUNS:
    for run in SMOKE_RUNS:
        print("\n" + "="*96)
        print(f"DIAGNOSTICS + GALLERY · {run['video_id']}")
        print("="*96)
        registry_path = run["registry_path"]
        run["dataframe"] = smoke_diagnostics(registry_path)
        preview_uniform_timeline(
            registry_path, max_items=CONFIG["smoke_static_sample"],
            cols=CONFIG["smoke_preview_columns"],
        )
        interactive_keyframe_gallery(
            registry_path,
            page_size=CONFIG["smoke_preview_page_size"],
            cols=CONFIG["smoke_preview_columns"],
        )


## 19. Deep inspection quanh timestamp nghi ngờ

`smoke_inspect_times` cho phép xem raw source frame quanh một hoặc nhiều timestamp và đo
khoảng cách đến retained keyframe gần nhất. Tất cả window được lấy trong **một lượt decode
tuần tự**, vì vậy `frame_idx` vẫn là decode-order identity thật và nhiều timestamp không làm
notebook decode lại từ đầu nhiều lần.


In [ ]:
def inspect_raw_windows(video_path, manifest, centers, radius_sec=2.0,
                        display_fps=5.0, max_side=960, cols=4):
    centers = sorted({float(x) for x in centers if float(x) >= 0})
    if not centers:
        print("Raw-window inspection disabled (smoke_inspect_times=[]).")
        return {}
    radius_sec = max(0.0, float(radius_sec))
    step = 1.0 / max(float(display_fps), 1e-6)
    windows = {
        center: {
            "start": max(0.0, center-radius_sec), "end": center+radius_sec,
            "next": max(0.0, center-radius_sec), "samples": [],
        }
        for center in centers
    }
    max_end = max(w["end"] for w in windows.values())

    container = av.open(str(video_path))
    stream = container.streams.video[0]; stream.thread_type = "AUTO"
    fps = probe_video(str(video_path)).fps
    decode_idx = 0
    for av_frame in container.decode(video=0):
        pts_time = av_frame_time(av_frame, decode_idx, fps)
        if pts_time > max_end + 1e-9:
            break
        for window in windows.values():
            if window["start"]-1e-9 <= pts_time <= window["end"]+1e-9 \
                    and pts_time+1e-9 >= window["next"]:
                bgr = av_frame.to_ndarray(format="bgr24")
                bgr = resize_max_side(bgr, int(max_side))
                window["samples"].append((float(pts_time), int(decode_idx), bgr))
                while window["next"] <= pts_time + 1e-9:
                    window["next"] += step
        decode_idx += 1
    container.close()

    retained_times = manifest["pts_time"].to_numpy(float) if len(manifest) else np.empty(0)
    retained_frames = set(manifest["frame_idx"].astype(int)) if len(manifest) else set()
    for center, window in windows.items():
        samples = window["samples"]
        print("\n" + "="*88)
        print(f"RAW INSPECTION {sec_to_hms(center)} ± {radius_sec:.1f}s | samples={len(samples)}")
        print("="*88)
        if not samples:
            print("Không có frame trong inspection window này.")
            continue
        rows = max(1, math.ceil(len(samples)/cols))
        fig, axes = plt.subplots(rows, cols, figsize=(4.2*cols, 3.1*rows))
        axes = np.atleast_1d(axes).reshape(-1)
        for ax in axes: ax.axis("off")
        for ax, (pts_time, frame_idx, bgr) in zip(axes, samples):
            nearest = float(np.min(np.abs(retained_times-pts_time))) if len(retained_times) else float("inf")
            tag = "✓ RETAINED" if frame_idx in retained_frames else f"nearest keyframe Δ={nearest:.3f}s"
            ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            ax.set_title(f"{sec_to_hms(pts_time)} · f{frame_idx}\n{tag}", fontsize=9)
            ax.axis("off")
        plt.tight_layout(); plt.show()
    return windows

if CONFIG["run_mode"] == "smoke" and SMOKE_RUNS:
    for run in SMOKE_RUNS:
        print("\n" + "#"*96)
        print(f"RAW-WINDOW REVIEW · {run['video_id']}")
        print("#"*96)
        manifest = run.get("dataframe")
        if manifest is None:
            manifest = pd.read_csv(run["registry_path"])
        run["inspection_windows"] = inspect_raw_windows(
            run["path"], manifest, CONFIG["smoke_inspect_times"],
            radius_sec=CONFIG["smoke_inspect_radius_sec"],
            display_fps=CONFIG["smoke_inspect_fps"],
            max_side=CONFIG["smoke_inspect_max_side"],
        )


## 20. Batch runner

Đổi `CONFIG["run_mode"]="batch"` và chọn category của ca bằng chuỗi CSV:

```python
"session_categories": "L21,L22,L23",
"session_partitions": "",       # mọi partition
"session_name": None,
```

Output ca này tự nằm tại `output_root/l21-l22-l23`. Ca kế tiếp chỉ cần đổi thành
`"L24,L25,L26"`; hai session không ghi đè nhau. Nếu một nhóm vẫn quá dài, dùng thêm
`range_start/range_end` hoặc `num_shards/shard_index`. Mỗi video ghi success marker
độc lập nên kernel chết giữa ca có thể chạy lại an toàn. Sau khi ca hoàn tất, Save
Version/commit toàn bộ thư mục session thành một Kaggle Dataset shard.

Để chỉ chạy `Videos_L26_d` và `Videos_L26_e`:

```python
"session_categories": "L26",
"session_partitions": "d,e",
```

Output tự mang suffix partition: `infoshootpp/l26__p-d-e/`.

Storage guard mặc định bắt đầu đóng gói từng video vào `archives/*.keyframes.zip`
khi session đạt 12 GiB. Archive được CRC-verify và registry được cập nhật trước khi
xoá loose JPEG; pass-1 cache của video thành công cũng được xoá. Vì JPEG đã nén, ZIP
chủ yếu giảm file-count. Hàng rào thật sự là tự dừng trước quota; output
`runs/session_state.json` sẽ in `next_range_start` để dùng cho session kế tiếp.


In [ ]:
GIB = 1024**3

def session_size_bytes(root=OUT):
    """Dùng du (nhanh với nhiều JPEG), fallback Python khi du không có."""
    root = Path(root)
    try:
        result = subprocess.run(
            ["du", "-sb", str(root)], capture_output=True, text=True, check=True
        )
        return int(result.stdout.split()[0])
    except Exception:
        total = 0
        for walk_root, _, files in os.walk(root):
            for filename in files:
                try: total += (Path(walk_root)/filename).stat().st_size
                except OSError: pass
        return total

def cleanup_completed_pass1_cache(video_id):
    if not CONFIG["session_storage"].get("delete_pass1_cache_after_success", True):
        return 0
    removed = 0
    for path in [OUT/"cache/probe"/f"{video_id}.npz",
                 OUT/"cache/sentinel"/f"{video_id}.npz"]:
        if path.exists():
            try:
                removed += path.stat().st_size
                path.unlink()
            except OSError as exc:
                log(video_id, f"WARNING: không xoá được completed cache {path}: {exc}")
    if removed:
        log(video_id, f"removed completed pass1 cache: {removed/GIB:.3f} GiB")
    return removed

def _atomic_write_registry(df, video_id):
    csv_path = OUT/"registry"/f"{video_id}.csv"
    jsonl_path = OUT/"registry"/f"{video_id}.jsonl"
    csv_tmp = csv_path.with_suffix(".csv.tmp")
    jsonl_tmp = jsonl_path.with_suffix(".jsonl.tmp")
    df.to_csv(csv_tmp, index=False)
    df.to_json(jsonl_tmp, orient="records", lines=True, force_ascii=False)
    os.replace(csv_tmp, csv_path); os.replace(jsonl_tmp, jsonl_path)

def archive_video_keyframes(video_id):
    """Archive một video, CRC-verify, cập nhật registry rồi mới xoá loose JPEG."""
    cfg = CONFIG["session_storage"]
    registry_path = OUT/"registry"/f"{video_id}.csv"
    if not registry_path.exists(): return None
    df = pd.read_csv(registry_path)
    if df.empty: return None
    category = str(df.category.iloc[0])
    image_dir = OUT/"keyframes"/category/video_id
    target = OUT/"archives"/f"{video_id}.keyframes.zip"
    meta_path = OUT/"archives"/f"{video_id}.archive.json"
    if not image_dir.exists():
        if target.exists() and meta_path.exists():
            return json.loads(meta_path.read_text(encoding="utf-8"))
        return None
    jpgs = sorted(image_dir.glob("*.jpg"))
    if not jpgs: return None

    members = {
        path.name: (Path("keyframes")/category/video_id/path.name).as_posix()
        for path in jpgs
    }
    original_bytes = sum(path.stat().st_size for path in jpgs)
    tmp = target.with_suffix(target.suffix+".tmp")
    tmp.unlink(missing_ok=True)
    with zipfile.ZipFile(
        tmp, "w", compression=zipfile.ZIP_DEFLATED,
        compresslevel=int(cfg.get("zip_compresslevel", 1)), allowZip64=True,
    ) as zf:
        for path in jpgs:
            zf.write(path, arcname=members[path.name])
    with zipfile.ZipFile(tmp, "r") as zf:
        bad_member = zf.testzip()
        archived_names = set(zf.namelist())
    expected_names = set(members.values())
    if bad_member is not None or archived_names != expected_names:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(
            f"Archive verify failed {video_id}: bad={bad_member}, "
            f"expected={len(expected_names)}, found={len(archived_names)}"
        )
    os.replace(tmp, target)

    archive_rel = target.relative_to(OUT).as_posix()
    archive_members, archive_uris = [], []
    for row in df.itertuples(index=False):
        old_path = getattr(row, "image_path", None)
        if old_path is None or pd.isna(old_path):
            archive_members.append(None); archive_uris.append(None); continue
        member = members.get(Path(str(old_path)).name)
        if member is None:
            raise RuntimeError(f"{video_id}: registry image không có trong archive")
        archive_members.append(member)
        archive_uris.append(f"zip://{target}!/{member}")
    df["image_archive_relpath"] = archive_rel
    df["image_archive_member"] = archive_members
    df["image_archive_path"] = str(target)
    if "image_path" in df: df["image_path"] = archive_uris
    _atomic_write_registry(df, video_id)

    archive_bytes = target.stat().st_size
    metadata = {
        "video_id":video_id, "category":category,
        "pipeline_signature":PIPELINE_SIGNATURE,
        "archive_relpath":archive_rel, "file_count":len(jpgs),
        "original_bytes":original_bytes, "archive_bytes":archive_bytes,
        "compression_ratio":archive_bytes/max(original_bytes,1),
        "verified":True, "created_at_unix":time.time(),
    }
    atomic_json(meta_path, metadata)
    if cfg.get("delete_images_after_verify", True):
        shutil.rmtree(image_dir)
        try: image_dir.parent.rmdir()
        except OSError: pass
    log(video_id, f"archived {len(jpgs):,} JPEG → {target.name} | "
        f"{original_bytes/GIB:.3f}→{archive_bytes/GIB:.3f} GiB")
    return metadata

def archive_completed_keyframes():
    archived = []
    for success in sorted((OUT/"registry").glob("*._SUCCESS.json")):
        video_id = success.name[:-len("._SUCCESS.json")]
        category = video_id.split("_",1)[0]
        if (OUT/"keyframes"/category/video_id).exists():
            metadata = archive_video_keyframes(video_id)
            if metadata is not None: archived.append(metadata)
    return archived

def storage_policy_after_video(video_id):
    cfg = CONFIG["session_storage"]
    if not cfg.get("enabled", True): return False, session_size_bytes()
    cleanup_completed_pass1_cache(video_id)
    used = session_size_bytes()
    trigger = float(cfg.get("archive_trigger_gb", 12.0))*GIB
    if cfg.get("archive_keyframes", True) and used >= trigger:
        archive_completed_keyframes()
        used = session_size_bytes()
    stop_line = (
        float(cfg.get("quota_gb", 20.0))
        - float(cfg.get("reserve_for_next_video_gb", 2.0))
    )*GIB
    should_stop = used >= max(0, stop_line)
    log(f"Session storage: {used/GIB:.3f} GiB | "
        f"next-video stop line={stop_line/GIB:.3f} GiB")
    return should_stop, used

def write_batch_state(rows, results, stopped_for_storage, next_position, used_bytes):
    failed = [
        str(x.get("video_id")) for x in results
        if str(x.get("status", "")).startswith("error:")
    ]
    registry_count = len([p for p in (OUT/"registry").glob("*.csv") if "__smoke" not in p.stem])
    success_count = len(list((OUT/"registry").glob("*._SUCCESS.json")))
    next_range_start = next_video_id = None
    if next_position is not None and next_position < len(rows):
        next_row = rows.iloc[int(next_position)]
        next_range_start = int(next_row["_scope_index"])
        next_video_id = str(next_row.video_id)
    session_complete = bool(
        not failed and registry_count == success_count
        and (stopped_for_storage or success_count == len(rows))
    )
    state = {
        "session_id":SESSION_ID,
        "session_categories":SESSION_CATEGORIES or list(CONFIG["categories"]),
        "session_partitions":SESSION_PARTITIONS or ["all"],
        "assigned_video_count":int(len(rows)),
        "attempted_video_count":int(len(results)),
        "completed_videos":success_count, "registry_csv":registry_count,
        "failed_video_ids":failed, "stopped_for_storage":bool(stopped_for_storage),
        "session_complete":session_complete,
        "mergeable_video_count":success_count if session_complete else 0,
        "used_bytes":int(used_bytes), "used_gib":used_bytes/GIB,
        "next_range_start":next_range_start, "next_video_id":next_video_id,
        "updated_at_unix":time.time(),
    }
    atomic_json(OUT/"runs"/"session_state.json", state)
    return state

def run_batch():
    assert_full_dataset_ready_for_batch()
    rows = selected_video_rows()
    if rows.empty:
        raise RuntimeError(f"Session {SESSION_ID} không có video nào sau range/shard")
    log(f"Session={SESSION_ID} | categories={SESSION_CATEGORIES or 'ALL'} | "
        f"partitions={SESSION_PARTITIONS or 'ALL'} | "
        f"batch={len(rows)} video | range={CONFIG['range_start']}:{CONFIG['range_end']} | "
        f"shard={CONFIG['shard_index']}/{CONFIG['num_shards']}")
    results = []
    run_name = f"run_{SESSION_ID}"
    summary_path = OUT / "runs" / f"{run_name}.csv"
    stopped_for_storage = False
    next_position = None
    storage_cfg = CONFIG["session_storage"]
    if storage_cfg.get("enabled", True):
        for success in sorted((OUT/"registry").glob("*._SUCCESS.json")):
            video_id = success.name[:-len("._SUCCESS.json")]
            cleanup_completed_pass1_cache(video_id)
    used_bytes = session_size_bytes()
    if (storage_cfg.get("enabled", True)
            and storage_cfg.get("archive_keyframes", True)
            and used_bytes >= float(storage_cfg.get("archive_trigger_gb", 12.0))*GIB):
        archive_completed_keyframes()
        used_bytes = session_size_bytes()
    preflight_line = (
        float(storage_cfg.get("quota_gb", 20.0))
        - float(storage_cfg.get("reserve_for_next_video_gb", 2.0))
    )*GIB
    if storage_cfg.get("enabled", True) and used_bytes >= max(0, preflight_line):
        stopped_for_storage = True; next_position = 0
        log(f"SESSION STORAGE STOP trước video đầu: {used_bytes/GIB:.3f} GiB")

    iterator = enumerate(rows.itertuples(index=False))
    for position, row in tqdm(iterator, total=0 if stopped_for_storage else len(rows), desc="videos"):
        if stopped_for_storage: break
        try:
            result = process_video(row.path, row.video_id, SELECTOR, FINAL_ENCODER)
        except Exception as exc:
            traceback.print_exc()
            result = {"video_id": row.video_id, "status": f"error:{type(exc).__name__}",
                      "error": str(exc)}
            atomic_json(OUT/"errors"/f"{row.video_id}.error.json", {
                **result, "path":row.path, "traceback":traceback.format_exc(),
            })
        results.append(result)
        pd.DataFrame(results).to_csv(summary_path, index=False)
        status = str(result.get("status", ""))
        if status in {"ok", "skip"}:
            (OUT/"errors"/f"{row.video_id}.error.json").unlink(missing_ok=True)
            stopped_for_storage, used_bytes = storage_policy_after_video(row.video_id)
            if stopped_for_storage and position+1 < len(rows):
                next_position = position+1
                log("SESSION STORAGE STOP:",
                    f"đã dùng {used_bytes/GIB:.3f} GiB; session kế tiếp đặt "
                    f"range_start={int(rows.iloc[next_position]['_scope_index'])} "
                    f"(next={rows.iloc[next_position].video_id})")
            elif stopped_for_storage:
                stopped_for_storage = False
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        write_batch_state(rows, results, stopped_for_storage, next_position, used_bytes)
        if stopped_for_storage: break
    summary = pd.DataFrame(results)
    state = write_batch_state(rows, results, stopped_for_storage, next_position, used_bytes)
    display(summary); display(pd.DataFrame([state]))
    return summary

BATCH_SUMMARY = None
if CONFIG["run_mode"] == "batch":
    BATCH_SUMMARY = run_batch()


## 21. Validation, thống kê compression và AIC-KeyframeBench

Annotation CSV tối thiểu: `video_id,event_start,event_end,event_type`. `clear_start/clear_end` là optional. Metric text dưới đây đo temporal hit; để đo *readable TextHitRecall* thật sự, chạy OCR trên retained JPEG rồi so transcript ground truth.


In [ ]:
def validate_outputs(video_ids=None):
    video_ids = video_ids or [p.stem for p in (OUT/"registry").glob("*.csv")]
    issues, stats = [], []
    for vid in video_ids:
        path = OUT/"registry"/f"{vid}.csv"
        if not path.exists(): issues.append((vid,"missing_registry")); continue
        df = pd.read_csv(path)
        if df.empty: issues.append((vid,"empty_registry")); continue
        if df.frame_idx.duplicated().any(): issues.append((vid,"duplicate_frame_idx"))
        if not df.frame_idx.is_monotonic_increasing: issues.append((vid,"not_sorted"))
        map_path = OUT/"map-keyframes"/f"{vid}.csv"
        expected_map_columns = ["n", "pts_time", "fps", "frame_idx"]
        if not map_path.exists():
            issues.append((vid,"missing_legacy_map"))
        else:
            map_df = pd.read_csv(map_path)
            if list(map_df.columns) != expected_map_columns:
                issues.append((vid,"bad_legacy_map_header"))
            elif len(map_df) != len(df):
                issues.append((vid,"legacy_map_row_count_mismatch"))
        archive_paths = (
            df.image_archive_path.dropna().astype(str).unique().tolist()
            if "image_archive_path" in df else []
        )
        if archive_paths:
            missing_archives = [p for p in archive_paths if not Path(p).exists()]
            if missing_archives:
                issues.append((vid,f"missing_archives:{len(missing_archives)}"))
        else:
            image_paths = df.image_path.dropna().astype(str) if "image_path" in df else []
            missing = [p for p in image_paths if not Path(p).exists()]
            if missing: issues.append((vid,f"missing_images:{len(missing)}"))
        duration = max(float(df.pts_time.max()), 1e-9)
        gaps = np.diff(df.pts_time.to_numpy(float)) if len(df) >= 2 else np.empty(0)
        max_gap = float(gaps.max()) if len(gaps) else 0.0
        allowed = CONFIG["coverage"]["max_gap_sec"] + CONFIG["coverage"]["verify_tolerance_sec"]
        conditional_ok = True
        if CONFIG["coverage"]["enabled"] and len(gaps):
            is_l25_static = (
                str(df.category.iloc[0]) == "L25"
                and "l25_static_run_id" in df and "l25_static_verified" in df
            )
            static_allowed = (
                CONFIG["l25_static_consolidation"]["max_static_gap_sec"]
                + CONFIG["coverage"]["verify_tolerance_sec"]
            )
            for i,gap in enumerate(gaps):
                same_static_run = bool(
                    is_l25_static
                    and df.iloc[i].l25_static_verified
                    and df.iloc[i+1].l25_static_verified
                    and int(df.iloc[i].l25_static_run_id) >= 0
                    and int(df.iloc[i].l25_static_run_id) == int(df.iloc[i+1].l25_static_run_id)
                )
                if gap > (static_allowed if same_static_run else allowed):
                    conditional_ok = False
                    issues.append((vid,f"coverage_gap:{float(gap):.3f}s@row{i}"))
                    break
        stats.append({"video_id":vid,"frames":len(df),"retained_fps":len(df)/duration,
                      "max_temporal_gap_sec":max_gap,
                      "conditional_coverage_ok":conditional_ok})
    return pd.DataFrame(issues, columns=["video_id","issue"]), pd.DataFrame(stats)

def evaluate_event_annotations(annotation_csv):
    ann = pd.read_csv(annotation_csv)
    required = {"video_id","event_start","event_end","event_type"}
    if not required.issubset(ann.columns):
        raise ValueError(f"Annotation thiếu cột: {sorted(required-set(ann.columns))}")
    rows = []
    registry_cache = {}
    for item in ann.itertuples(index=False):
        vid = canonical_video_id(item.video_id)
        if vid not in registry_cache:
            path = OUT/"registry"/f"{vid}.csv"
            registry_cache[vid] = pd.read_csv(path) if path.exists() else pd.DataFrame()
        reg = registry_cache[vid]
        times = reg.pts_time.to_numpy() if len(reg) else np.array([])
        hit = bool(np.any((times >= float(item.event_start)) & (times <= float(item.event_end))))
        clear_hit = None
        if hasattr(item,"clear_start") and hasattr(item,"clear_end") and pd.notna(item.clear_start) and pd.notna(item.clear_end):
            clear_hit = bool(np.any((times >= float(item.clear_start)) & (times <= float(item.clear_end))))
        rows.append({"video_id":vid,"event_type":item.event_type,
                     "duration":float(item.event_end)-float(item.event_start),
                     "hit":hit,"clear_hit":clear_hit})
    detail = pd.DataFrame(rows)
    summary = detail.groupby("event_type").agg(events=("hit","size"), event_hit_recall=("hit","mean"),
                                                clear_hit_recall=("clear_hit","mean")).reset_index()
    complete = detail.groupby("video_id").hit.all().mean() if len(detail) else np.nan
    print("Complete Video Recall:", complete)
    display(summary)
    return detail, summary

ISSUES, OUTPUT_STATS = validate_outputs()
display(ISSUES)
if len(OUTPUT_STATS): display(OUTPUT_STATS.describe())


## 22. Hoàn tất session hoặc merge nhiều session

### Extraction session

Cell tạo `run_manifest.json`, global registry riêng của session và metadata ZIP. ZIP
không chứa JPEG/cache/embeddings; hãy commit **toàn bộ thư mục session** thành Kaggle
Dataset để giữ keyframe assets.

### Merge session

Tạo notebook/session mới, attach tất cả Kaggle Dataset shard rồi đặt:

```python
"run_mode": "merge",
"merge": {
    "input_roots": [
        "/kaggle/input/infoshootpp-l21-l23",
        "/kaggle/input/infoshootpp-l24-l26",
        "/kaggle/input/infoshootpp-l27-l30",
    ],
    "output_root": "/kaggle/working/infoshootpp_merged",
    "copy_assets": False,
    "strict_pipeline_signature": True,
    "strict_session_complete": True,
    "expected_categories": None,
    "expected_total_videos": 1478,
    "enforce_expected_total": True,
}
```

`copy_assets=False` là chế độ khuyên dùng: global registry có thêm
`source_session_id`, `source_session_root`, `asset_relpath` và trỏ trực tiếp tới JPEG
trong các attached shard, tránh copy hàng trăm GB. Đặt `True` chỉ khi output quota đủ;
khi đó loose JPEG hoặc ZIP archive cùng embedding được gom vật lý vào
`merge.output_root` (ZIP không tự giải nén).


In [ ]:
def merge_registry_outputs(root=OUT, include_smoke=False):
    root = Path(root)
    frames, summaries = [], []
    for path in sorted((root/"registry").glob("*.csv")):
        if not include_smoke and "__smoke" in path.stem: continue
        try: frames.append(pd.read_csv(path))
        except Exception as exc: log("skip registry",path,exc)
    for path in sorted((root/"registry").glob("*.stats.json")):
        if not include_smoke and "__smoke" in path.stem: continue
        try: summaries.append(json.loads(path.read_text(encoding="utf-8")))
        except Exception: pass
    registry = pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()
    summary = pd.DataFrame(summaries)
    if len(registry):
        registry.to_parquet(root/"frame_registry.parquet",index=False)
        registry.to_json(root/"frame_registry.jsonl",orient="records",lines=True,force_ascii=False)
    if len(summary): summary.to_csv(root/"run_summary.csv",index=False)
    print("Merged:",len(summary),"videos,",len(registry),"frames")
    return summary,registry

def write_run_manifest(root=OUT):
    root = Path(root)
    jpg_files = total_bytes = 0
    for walk_root, _, files in os.walk(root):
        for filename in files:
            path = Path(walk_root) / filename
            try: total_bytes += path.stat().st_size
            except OSError: pass
            if path.suffix.lower() == ".jpg": jpg_files += 1
    state_path = root/"runs"/"session_state.json"
    batch_state = (
        json.loads(state_path.read_text(encoding="utf-8"))
        if state_path.exists() else {}
    )
    archive_files = list((root/"archives").glob("*.keyframes.zip"))
    payload = {
        "manifest_type": "session",
        "session_id": SESSION_ID,
        "session_categories": SESSION_CATEGORIES or list(CONFIG["categories"]),
        "session_partitions": SESSION_PARTITIONS or ["all"],
        "selected_video_count": int(len(selected_video_rows())),
        "range_start": CONFIG["range_start"], "range_end": CONFIG["range_end"],
        "num_shards": CONFIG["num_shards"], "shard_index": CONFIG["shard_index"],
        "pipeline_version": CONFIG["version"],
        "pipeline_signature": PIPELINE_SIGNATURE,
        "config": CONFIG,
        "created_at_unix": time.time(),
        "completed_videos": len(list((root/"registry").glob("*._SUCCESS.json"))),
        "registry_csv": len([p for p in (root/"registry").glob("*.csv") if "__smoke" not in p.stem]),
        "session_complete": bool(batch_state.get("session_complete", False)),
        "mergeable_video_count": int(batch_state.get("mergeable_video_count", 0)),
        "stopped_for_storage": bool(batch_state.get("stopped_for_storage", False)),
        "next_range_start": batch_state.get("next_range_start"),
        "next_video_id": batch_state.get("next_video_id"),
        "jpg_files": jpg_files,
        "keyframe_archives": len(archive_files),
        "archive_bytes": sum(path.stat().st_size for path in archive_files),
        "total_bytes": total_bytes,
    }
    atomic_json(root/"run_manifest.json", payload)
    print(json.dumps({k:v for k,v in payload.items() if k != "config"}, indent=2))
    return payload

def export_registry_bundle(root=OUT, label=None):
    if not CONFIG["storage"]["export_registry_zip"]: return None
    root = Path(root)
    label = re.sub(r"[^a-zA-Z0-9._-]+", "-", label or SESSION_ID).strip("-_")
    target = Path("/kaggle/working") / f"infoshootpp_v4_6_{label}_registry.zip"
    roots = [root/"registry",root/"map-keyframes",root/"runs",root/"errors"]
    top_files = [root/"frame_registry.parquet",root/"frame_registry.jsonl",
                 root/"run_summary.csv",root/"run_manifest.json",
                 root/"merge_manifest.json",root/"asset_index.csv"]
    with zipfile.ZipFile(target,"w",compression=zipfile.ZIP_DEFLATED) as zf:
        for item_root in roots:
            if not item_root.exists(): continue
            for path in item_root.rglob("*"):
                if path.is_file(): zf.write(path,path.relative_to(root))
        for path in sorted((root/"archives").glob("*.archive.json")):
            zf.write(path, path.relative_to(root))
        for path in top_files:
            if path.exists(): zf.write(path,path.relative_to(root))
    print("Registry bundle:",target)
    return target

def discover_merge_roots(values):
    if isinstance(values, (str, Path)):
        values = [values]
    roots = []
    for raw in values or []:
        path = Path(raw)
        if not path.exists():
            raise FileNotFoundError(f"Merge input không tồn tại: {path}")
        manifests = [path/"run_manifest.json"] if (path/"run_manifest.json").exists() else sorted(
            path.rglob("run_manifest.json")
        )
        if not manifests:
            raise FileNotFoundError(f"Không tìm thấy run_manifest.json dưới {path}")
        roots.extend(x.parent for x in manifests)
    unique = []
    for root in roots:
        resolved = str(root.resolve())
        if resolved not in {str(x.resolve()) for x in unique}:
            unique.append(root)
    if not unique:
        raise ValueError("merge.input_roots đang rỗng")
    return unique

def merge_session_outputs():
    cfg = CONFIG["merge"]
    roots = discover_merge_roots(cfg.get("input_roots"))
    destination = Path(cfg["output_root"])
    destination.mkdir(parents=True, exist_ok=True)
    for name in ["keyframes", "archives", "registry", "map-keyframes", "selector-embeddings",
                 "final-embeddings", "runs", "errors"]:
        (destination/name).mkdir(parents=True, exist_ok=True)

    sessions, entries = [], {}
    signatures = set()
    for root in roots:
        manifest = json.loads((root/"run_manifest.json").read_text(encoding="utf-8"))
        if manifest.get("manifest_type", "session") != "session":
            raise ValueError(f"Input không phải extraction session: {root}")
        session_id = str(manifest.get("session_id") or root.name)
        signature = manifest.get("pipeline_signature")
        signatures.add(signature)
        sessions.append({"session_id":session_id, "root":str(root), "manifest":manifest})
        registry_files = [p for p in sorted((root/"registry").glob("*.csv")) if "__smoke" not in p.stem]
        if not registry_files:
            raise RuntimeError(f"Session không có production registry CSV: {root}")
        if cfg.get("strict_session_complete", True):
            session_complete = bool(manifest.get("session_complete", False))
            expected_session = int(manifest.get("mergeable_video_count", -1))
            completed_session = int(manifest.get("completed_videos", -1))
            if (not session_complete or expected_session < 0
                    or len(registry_files) != expected_session
                    or completed_session != expected_session):
                raise RuntimeError(
                    f"Session chưa sẵn sàng merge: {session_id} | "
                    f"session_complete={session_complete}, mergeable={expected_session}, "
                    f"registry={len(registry_files)}, success={completed_session}"
                )
        for registry_path in registry_files:
            df = pd.read_csv(registry_path)
            if df.empty or "video_id" not in df:
                raise ValueError(f"Registry rỗng/thiếu video_id: {registry_path}")
            video_ids = df.video_id.astype(str).unique().tolist()
            if len(video_ids) != 1:
                raise ValueError(f"Registry chứa nhiều video_id: {registry_path}")
            video_id = video_ids[0]
            if video_id in entries:
                old = entries[video_id]
                raise RuntimeError(
                    f"Duplicate video_id giữa session: {video_id}\n"
                    f"- {old['root']}\n- {root}"
                )
            entries[video_id] = {
                "root":root, "session_id":session_id,
                "registry_path":registry_path, "df":df,
            }

    if cfg.get("strict_pipeline_signature", True):
        if None in signatures or len(signatures) != 1:
            raise RuntimeError(f"Pipeline signature giữa session không đồng nhất: {signatures}")
        only_signature = next(iter(signatures))
        if only_signature != PIPELINE_SIGNATURE:
            raise RuntimeError(
                f"Session signature {only_signature} != notebook merge {PIPELINE_SIGNATURE}"
            )

    categories_found = sorted({str(x) for e in entries.values() for x in e["df"].category.unique()})
    expected_category_spec = cfg.get("expected_categories")
    expected_categories = (
        parse_category_spec(expected_category_spec)
        if expected_category_spec is not None else list(CONFIG["categories"])
    )
    missing_categories = sorted(set(expected_categories)-set(categories_found))
    unexpected_categories = sorted(set(categories_found)-set(expected_categories))
    expected = cfg.get("expected_total_videos")
    count_ok = expected is None or len(entries) == int(expected)
    categories_ok = not missing_categories and not unexpected_categories
    audit_ok = bool(count_ok and categories_ok)
    audit = {
        "videos":len(entries), "expected":expected, "count_ok":count_ok,
        "expected_categories":expected_categories, "categories_found":categories_found,
        "missing_categories":missing_categories,
        "unexpected_categories":unexpected_categories,
        "audit_ok":audit_ok,
    }
    print("MERGE AUDIT:", json.dumps(audit, ensure_ascii=False, indent=2))
    if cfg.get("enforce_expected_total", True) and not audit_ok:
        raise RuntimeError("MERGE BLOCKED: chưa đủ video/category hoặc có shard bị thiếu")

    copy_assets = bool(cfg.get("copy_assets", False))
    merged_frames, stats_rows, asset_rows = [], [], []
    for session in sessions:
        session_id, root = session["session_id"], Path(session["root"])
        for folder in ["runs", "errors"]:
            for source in sorted((root/folder).glob("*")):
                if source.is_file():
                    shutil.copy2(
                        source,
                        destination/folder/f"{session_id}__{source.name}",
                    )
    for video_id in sorted(entries, key=natural_video_key):
        entry = entries[video_id]
        root, session_id = entry["root"], entry["session_id"]
        df = entry["df"].copy()
        categories = df.category.astype(str).unique().tolist()
        if len(categories) != 1:
            raise ValueError(f"{video_id}: registry chứa nhiều category")
        category = categories[0]
        relative_dir = Path("keyframes")/category/video_id
        source_image_dir = root/relative_dir
        archive_rel_values = (
            df.image_archive_relpath.dropna().astype(str).unique().tolist()
            if "image_archive_relpath" in df else []
        )
        if len(archive_rel_values) > 1:
            raise ValueError(f"{video_id}: registry trỏ tới nhiều keyframe archive")
        archive_rel = Path(archive_rel_values[0]) if archive_rel_values else None
        source_archive = root/archive_rel if archive_rel is not None else None
        has_archive = source_archive is not None
        if has_archive and not source_archive.exists():
            raise FileNotFoundError(f"Thiếu keyframe archive: {source_archive}")
        has_images = "image_path" in df and df.image_path.notna().any()
        if has_images and not source_image_dir.exists() and not has_archive:
            raise FileNotFoundError(f"Thiếu keyframe directory/archive: {source_image_dir}")
        if copy_assets and source_image_dir.exists():
            shutil.copytree(source_image_dir, destination/relative_dir, dirs_exist_ok=True)
        archive_target = None
        if has_archive:
            archive_target = destination/archive_rel if copy_assets else source_archive
            if copy_assets:
                archive_target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source_archive, archive_target)
                archive_meta_source = root/"archives"/f"{video_id}.archive.json"
                if archive_meta_source.exists():
                    shutil.copy2(archive_meta_source, destination/"archives"/archive_meta_source.name)

        asset_relpaths, merged_image_paths = [], []
        for row in df.itertuples(index=False):
            old_path = getattr(row, "image_path", None)
            if old_path is None or pd.isna(old_path):
                asset_relpaths.append(None); merged_image_paths.append(None); continue
            archive_member = getattr(row, "image_archive_member", None)
            if has_archive and archive_member is not None and not pd.isna(archive_member):
                relative_path = Path(str(archive_member))
                asset_relpaths.append(relative_path.as_posix())
                merged_image_paths.append(
                    f"zip://{archive_target}!/{relative_path.as_posix()}"
                )
                continue
            filename = Path(str(old_path)).name
            relative_path = relative_dir/filename
            asset_relpaths.append(relative_path.as_posix())
            target_root = destination if copy_assets else root
            merged_image_paths.append(str(target_root/relative_path))
        df["source_session_id"] = session_id
        df["source_session_root"] = str(root)
        df["asset_relpath"] = asset_relpaths
        if has_archive:
            df["image_archive_relpath"] = archive_rel.as_posix()
            df["image_archive_path"] = str(archive_target)
        if "image_path" in df: df["image_path"] = merged_image_paths
        df.to_csv(destination/"registry"/f"{video_id}.csv", index=False)
        df.to_json(destination/"registry"/f"{video_id}.jsonl", orient="records",
                   lines=True, force_ascii=False)
        merged_frames.append(df)

        map_source = root/"map-keyframes"/f"{video_id}.csv"
        if map_source.exists():
            shutil.copy2(map_source, destination/"map-keyframes"/map_source.name)
        else:
            df[["n","pts_time","fps","frame_idx"]].to_csv(
                destination/"map-keyframes"/f"{video_id}.csv", index=False
            )
        stats_source = root/"registry"/f"{video_id}.stats.json"
        if stats_source.exists():
            stats = json.loads(stats_source.read_text(encoding="utf-8"))
            stats["source_session_id"] = session_id
            stats_rows.append(stats)
            atomic_json(destination/"registry"/stats_source.name, stats)
        success_source = root/"registry"/f"{video_id}._SUCCESS.json"
        if success_source.exists():
            shutil.copy2(success_source, destination/"registry"/success_source.name)

        selector_source = root/"selector-embeddings"/f"{video_id}.npy"
        final_source = root/"final-embeddings"/f"{video_id}.npy"
        if copy_assets:
            if selector_source.exists(): shutil.copy2(selector_source, destination/"selector-embeddings"/selector_source.name)
            if final_source.exists(): shutil.copy2(final_source, destination/"final-embeddings"/final_source.name)
        selector_target = (
            destination/"selector-embeddings"/selector_source.name
            if copy_assets and selector_source.exists() else selector_source
        )
        final_target = (
            destination/"final-embeddings"/final_source.name
            if copy_assets and final_source.exists() else final_source
        )
        asset_rows.append({
            "video_id":video_id, "category":category,
            "source_session_id":session_id, "source_session_root":str(root),
            "keyframe_dir":str((destination if copy_assets else root)/relative_dir),
            "keyframe_archive_path":str(archive_target) if has_archive else None,
            "selector_embedding_path":str(selector_target) if selector_source.exists() else None,
            "final_embedding_path":str(final_target) if final_source.exists() else None,
        })

    registry = pd.concat(merged_frames, ignore_index=True).sort_values(
        ["category","video_id","frame_idx"], kind="stable"
    ).reset_index(drop=True)
    if registry.duplicated(["video_id","frame_idx"]).any():
        raise RuntimeError("Merged registry có duplicate (video_id, frame_idx)")
    summary = pd.DataFrame(stats_rows)
    registry.to_parquet(destination/"frame_registry.parquet", index=False)
    registry.to_json(destination/"frame_registry.jsonl", orient="records",
                     lines=True, force_ascii=False)
    if len(summary): summary.to_csv(destination/"run_summary.csv", index=False)
    pd.DataFrame(asset_rows).to_csv(destination/"asset_index.csv", index=False)

    merge_manifest = {
        "manifest_type":"merge", "pipeline_version":CONFIG["version"],
        "pipeline_signature":PIPELINE_SIGNATURE, "created_at_unix":time.time(),
        "copy_assets":copy_assets, "source_sessions":[
            {"session_id":x["session_id"], "root":x["root"]} for x in sessions
        ],
        "audit":audit, "merged_frames":int(len(registry)),
        "output_root":str(destination),
    }
    atomic_json(destination/"merge_manifest.json", merge_manifest)
    print(f"Merged {len(entries):,} videos / {len(registry):,} keyframes → {destination}")
    display(registry.groupby("category").video_id.nunique().rename("videos").reset_index())
    return summary, registry, merge_manifest

RUN_MANIFEST = MERGE_MANIFEST = None
if CONFIG["run_mode"] == "merge":
    MERGED_SUMMARY, MERGED_REGISTRY, MERGE_MANIFEST = merge_session_outputs()
    REGISTRY_ZIP = export_registry_bundle(MERGE_OUT, label="merged")
else:
    MERGED_SUMMARY, MERGED_REGISTRY = merge_registry_outputs(OUT, include_smoke=False)
    RUN_MANIFEST = write_run_manifest(OUT)
    REGISTRY_ZIP = export_registry_bundle(OUT, label=SESSION_ID)
